<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_11_Hybrid_Neyman_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 11 — A response-matched, simulator-calibrated Neyman construction

Exercise 5 ended with pseudo-experiments from the learned hNDE likelihood. Here we use that likelihood as an ordering model, correct its leading simulator response mismatch, amortize the resulting sampling law over the physical signal strength $\mu$, and calibrate the remaining discrepancy with simulator pseudo-experiments.

The notebook deliberately separates six statistical objects:

1. the **frozen event-level hNDE likelihood** $L_{\rm H}(\nu)$, whose internal signal-strength coordinate is denoted $\nu$;
2. a monotone **pseudo-truth response map** $g(\mu)$ from physical simulator truth to the KL-optimal hNDE coordinate;
3. the response-corrected ordering statistic $t_\mu^{(g)}=-2\log[L_{\rm H}(g(\mu))/L_{\rm H}(\widehat\nu)]$;
4. a conditional spline plus a first density-ratio correction trained on response-matched hNDE toys;
5. a PIT-space simulator correction trained on a finite construction reservoir, with extra proposal density at low $\mu$;
6. coverage diagnostics with distinct roles: a same-template algorithmic control, construction-template bootstraps, and a final audit built from an event-level simulator reservoir that was never used by Exercise 5 or by this construction.

The response map removes the large pseudo-truth displacement but does not assert that the hNDE likelihood is correct. Sandwich-variance changes, skewness, discreteness, and tail differences remain for the PIT ratio to learn. Exact coverage is assessed only after all maps and cutoffs have been frozen.

The expensive Exercise 5 event networks are loaded from their checkpoints. Toy ensembles are generated in resumable shards. All response-dependent caches use new versioned paths: the old statistic flow, first ratio, PIT ratio, endpoint toys, and auditor are statistically incompatible with this construction.


In [ ]:
## ==========================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ==========================================================================
import os, sys

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "iminuit",
    "mplhep",
    "nflows",
    "pyarrow",
]


def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
    if REMAKE_EVENTS or not Path("dataframes/signal.parquet").exists():
        run(
            sys.executable,
            TUTORIAL_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
        )

print("Working dir:", os.getcwd())


## Statistical construction at a glance

Let $\mu$ denote the physical parameter that labels the simulator law, and let $\nu$ denote the coordinate of the frozen hNDE likelihood. Under simulator misspecification, the hNDE MLE need not converge to $\mu$. Its population target is instead

$$
g(\mu)
=\arg\max_{\nu\geq0}
\mathbb E_{\mathcal D\sim p_{\rm sim}(\cdot\mid\mu)}
[\log L_{\rm H}(\nu;\mathcal D)].
$$

We therefore use the response-matched ordering statistic

$$
t_\mu^{(g)}(\mathcal D)
=-2\log\frac{L_{\rm H}(g(\mu);\mathcal D)}
                 {L_{\rm H}(\widehat\nu;\mathcal D)}
\geq0.
$$

The physical Neyman parameter remains $\mu$ everywhere: flow contexts, classifier inputs, cutoff curves, the auditor, and confidence-set inversion all use $\mu$. Only the hNDE generation coordinate and the likelihood numerator use $g(\mu)$.

Response-matched hNDE toys are generated at $\nu=g(\mu)$ and evaluated with the same numerator. A conditional spline $q_\phi$ and a matched classifier learn their density. In $y=\log(t^{(g)}+\epsilon)$,

$$
C_\phi(\mu)=\int_{\log\epsilon}^{\infty}q_\phi(v\mid\mu)\,dv,
\qquad
\widetilde q_\phi(y\mid\mu)
=\frac{q_\phi(y\mid\mu)\,\mathbb I[y\geq\log\epsilon]}
       {C_\phi(\mu)},
$$

and

$$
p_{\rm H}^{(g)}(y\mid\mu)
=\frac{\widetilde q_\phi(y\mid\mu)r_1(y,\mu)}
       {\widetilde Z_1(\mu)},
\qquad
F_{\rm H}^{(g)}(t\mid\mu)
=\int_{\log\epsilon}^{\log(t+\epsilon)}
p_{\rm H}^{(g)}(y\mid\mu)\,dy.
$$

A simulator toy generated at physical truth $\mu$ is evaluated with $t_\mu^{(g)}$ and mapped to

$$
u_0=F_{\rm H}^{(g)}(t_\mu^{(g)}\mid\mu)\in[0,1].
$$

If response matching had removed every simulator discrepancy, $u_0\mid\mu$ would be uniform. The second matched classifier estimates

$$
r_{\rm cal}(u,\mu)
=\frac{p_{\rm sim}^{U_0}(u\mid\mu)}{\mathrm{Uniform}(u)}
=p_{\rm sim}^{U_0}(u\mid\mu).
$$

Finite neural odds are normalized explicitly,

$$
G(u\mid\mu)
=\frac{\int_0^u r_{\rm cal}(v,\mu)\,dv}
       {\int_0^1 r_{\rm cal}(v,\mu)\,dv},
$$

giving

$$
F_{\rm cal}(t\mid\mu)
=G\!\left(F_{\rm H}^{(g)}(t\mid\mu)\mid\mu\right),
\qquad
U_{\rm cal}=F_{\rm cal}(T_\mu^{(g)}\mid\mu).
$$

In the oracle continuous limit, $U_{\rm cal}\mid\mu\sim\mathrm{Uniform}(0,1)$. Because large $t_\mu^{(g)}$ rejects the tested value, the 95% acceptance rule is $U_{\rm cal}\leq0.95$. The PIT convention is the upper-CDF convention appropriate to a likelihood-ratio statistic; it reverses the lower-quantile sign convention used in the [LF2I paper](https://arxiv.org/abs/2107.03920).


In [ ]:
import gc
import hashlib
import os
from pathlib import Path

# JAX fits the toy batches before PyTorch trains the large spline. Avoid
# reserving the whole Colab GPU so both frameworks can share it.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.stats import chi2, norm

import torch

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_hnpe import (
    train_ratio_classifier,
    train_spline_flow,
)
from utils_neyman import (
    asimov_test_statistic,
    binned_coverage,
    build_compressed_q_model,
    conditional_cdf_values,
    conditional_density_grid,
    conditional_quantiles,
    conditional_ratio_grid,
    conditional_row_quantiles,
    conservative_empirical_quantile,
    coverage_auditor_probability,
    run_cached_toy_ensemble,
    sample_truncated_spline_flow,
    simulator_templates_from_exercise5,
    train_coverage_auditor,
    wilson_interval,
)
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    collect_preselected_eval_rows,
    deterministic_row_priority,
    flow_sample_x,
    load_flow,
)
from utils_plotting import export_standalone_figure_script

FEATURES = list(FEATURES)
SEED = 11082026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Configuration and frozen Exercise 5 inputs

`FAST_MODE=True` is a structural test. The default run uses 500,000 response-matched hNDE toys, 100,000 uniform simulator-calibration toys, 50,000 additional low-$\mu$ calibration toys, a same-template internal audit, and a separate sealed event-reservoir audit.

The run tags are intentionally new. Once $g(\mu)$ changes, every cached statistic, flow, ratio, endpoint quantile, and auditor from the identity-response construction is stale. Checkpoint data fingerprints provide a second guard against accidental reuse.

Exercise 5 retained only the 250,000 smallest deterministic-priority evaluation rows per process. Its raw parquet files contain many additional PRESEL-passing rows in the same evaluation partition. Exercise 11 reconstructs the retained set exactly, proves that its complement is disjoint, saves the complement as a sealed audit template, and does not load that template until the final construction has been frozen.

The statistic flow keeps the Exercise 5 rational-quadratic-spline settings: ten spline transforms, 16 bins, tail bound 5, four residual blocks, and width 1024. For a scalar target, `utils_hnpe` uses a one-dimensional autoregressive spline; a coupling layer would otherwise have no second coordinate to transform.


In [ ]:
BASE_PATH = Path("dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
HYBRID_DENSITY_DIR = Path("saved_densities_hybrid")
EXERCISE5_RATIO_NORMALIZATION_PATH = (
    HYBRID_DENSITY_DIR / "ratio_normalization.npz"
)
if EXERCISE5_RATIO_NORMALIZATION_PATH.exists():
    with np.load(EXERCISE5_RATIO_NORMALIZATION_PATH) as saved_norm:
        EXERCISE5_RATIO_NORMALIZATION = {
            "signal": float(saved_norm["signal"]),
            "background": float(saved_norm["background"]),
        }
    EXERCISE5_NORMALIZATION_SOURCE = str(
        EXERCISE5_RATIO_NORMALIZATION_PATH
    )
else:
    # Legacy Exercise 5 saved normalized ratio arrays but not these two
    # finite-reference means. These values are transcribed from the
    # committed full-run output (printed to six decimal places).
    EXERCISE5_RATIO_NORMALIZATION = {
        "signal": 0.999149,
        "background": 0.999669,
    }
    EXERCISE5_NORMALIZATION_SOURCE = "committed Exercise 5 output"

FAST_MODE = False
LOAD_IF_AVAILABLE = True
BASE_RUN_TAG = "fast_response_v2" if FAST_MODE else "full_response_v2"
PIT_RUN_TAG = (
    "fast_pit_response_v2" if FAST_MODE else "full_pit_response_v2"
)
MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / BASE_RUN_TAG
CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / BASE_RUN_TAG
PIT_MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
PIT_CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
PLOT_DIR = Path("plots_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
SIMULATOR_TEMPLATE_DIR = Path(
    "saved_exercise11_simulator_reservoir_v2"
)
CONSTRUCTION_TEMPLATE_PATH = (
    SIMULATOR_TEMPLATE_DIR / "construction_template.npz"
)
SEALED_AUDIT_TEMPLATE_PATH = (
    SIMULATOR_TEMPLATE_DIR / "sealed_unused_event_audit_template.npz"
)
FIGURE_SCRIPT_DIR = Path("exercise11_figures_scripts")
RATIO1_MODEL_DIR = MODEL_DIR / "hnde_residual_ensemble4"
CALIBRATION_RATIO_MODEL_DIR = PIT_MODEL_DIR / "pit_calibration_ensemble4"
for directory in [
    MODEL_DIR, CACHE_DIR, PIT_MODEL_DIR, PIT_CACHE_DIR,
    PLOT_DIR, FIGURE_SCRIPT_DIR, SIMULATOR_TEMPLATE_DIR,
    RATIO1_MODEL_DIR, CALIBRATION_RATIO_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536
RATIO_ENSEMBLE_SIZE = 4
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12

MU_RANGE = (0.0, 3.0)
LOW_MU_CALIBRATION_RANGE = (0.0, 0.75)
EXERCISE5_EVAL_CAP = 250_000
EXERCISE5_EVAL_PRIORITY_SEED = {
    "signal": 12_447,
    "background": 12_548,
}
EXERCISE5_EXPECTED_SELECTED_EVAL = {
    "signal": 442_938,
    "background": 610_072,
}
TOY_Q_BINS = 512
TOY_BATCH_SIZE = 2_000
TOY_NEWTON_STEPS = 16
TOY_MU_MAX = 12.0
TOY_FIT_FINGERPRINT = "jax_newton16_bisection64_mumax12_v1"
T_OFFSET = 1.0e-6
LOG_RATIO_CLIP = 15.0
PIT_EPS = 1.0e-6
QUANTILE_LEVELS = np.asarray([0.50, 0.68, 0.90, 0.95, 0.99])
ANCHOR_MUS = np.asarray([0.0, 3.0])

if FAST_MODE:
    N_REFERENCE_EVENTS = 250_000
    N_HNDE_TOYS = 50_000
    N_FLOW_TOYS = 40_000
    N_RATIO1_TOYS = 10_000
    N_SIMULATOR_CALIBRATION_TOYS = 20_000
    N_SIMULATOR_LOW_MU_TOYS = 10_000
    N_SIMULATOR_INTERNAL_AUDIT_TOYS = 20_000
    N_SIMULATOR_AUDIT_TOYS = 20_000
    N_TEMPLATE_BOOTSTRAPS = 4
    N_TOYS_PER_TEMPLATE_BOOTSTRAP = 2_000
    RESPONSE_GRID_POINTS = 301
    N_ANCHOR_TOYS = 5_000
    N_AUDIT_ANCHOR_TOYS = 5_000
    FLOW_EPOCHS = 8
    RATIO_EPOCHS = 12
    QUADRATURE_MU_POINTS = 101
    QUADRATURE_Y_POINTS = 1_025
    PIT_QUADRATURE_POINTS = 1_025
else:
    N_REFERENCE_EVENTS = 5_000_000
    N_HNDE_TOYS = 500_000
    N_FLOW_TOYS = 400_000
    N_RATIO1_TOYS = 100_000
    N_SIMULATOR_CALIBRATION_TOYS = 100_000
    N_SIMULATOR_LOW_MU_TOYS = 50_000
    N_SIMULATOR_INTERNAL_AUDIT_TOYS = 100_000
    N_SIMULATOR_AUDIT_TOYS = 100_000
    N_TEMPLATE_BOOTSTRAPS = 20
    N_TOYS_PER_TEMPLATE_BOOTSTRAP = 5_000
    RESPONSE_GRID_POINTS = 1_001
    N_ANCHOR_TOYS = 25_000
    N_AUDIT_ANCHOR_TOYS = 25_000
    FLOW_EPOCHS = 70
    RATIO_EPOCHS = 50
    QUADRATURE_MU_POINTS = 301
    QUADRATURE_Y_POINTS = 4_097
    PIT_QUADRATURE_POINTS = 4_097

if N_FLOW_TOYS + N_RATIO1_TOYS != N_HNDE_TOYS:
    raise ValueError("The disjoint hNDE flow/ratio splits must exhaust the toys.")

STATISTIC_FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
STATISTIC_FLOW_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-4,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-7,
    "weight_decay": 0.0,
    "validation_fraction": 0.20,
    "patience": 5,
    "gradient_clip": 5.0,
}
CORRECTION_MODEL_CONFIG = {
    "hidden_features": 1024,
    "hidden_layers": 4,
    "dropout_probability": 0.0,
}
CORRECTION_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": RATIO_EPOCHS,
    "learning_rate": 1.0e-3,
    "lr_scheduler": "step",
    "lr_scheduler_factor": 0.01,
    "lr_scheduler_patience": 10,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}
AUDITOR_MODEL_CONFIG = {"hidden_features": 64, "hidden_layers": 3}
AUDITOR_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": 200 if not FAST_MODE else 40,
    "learning_rate": 1.0e-3,
    "weight_decay": 1.0e-4,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 5,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}


def export_exercise11_figure(fig, script_name):
    fig.savefig(PLOT_DIR / f"{script_name}.png", dpi=160)
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
    HYBRID_DENSITY_DIR / "weights_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend([
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        ])
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Exercise 11 loads the frozen Exercise 5 model and held-out arrays. "
        "Run Exercise 5 through 'Reconstruct and validate the hybrid densities' "
        "first. Missing:\n" + "\n".join(f"  - {path}" for path in missing_paths)
    )

print(f"Base/PIT run tags: {BASE_RUN_TAG} / {PIT_RUN_TAG}")
print(f"hNDE toys: {N_HNDE_TOYS:,}")
print(
    "simulator uniform/low-mu calibration toys: "
    f"{N_SIMULATOR_CALIBRATION_TOYS:,}/{N_SIMULATOR_LOW_MU_TOYS:,}"
)
print(
    "internal/sealed audit toys: "
    f"{N_SIMULATOR_INTERNAL_AUDIT_TOYS:,}/{N_SIMULATOR_AUDIT_TOYS:,}"
)
print(f"Standalone figure scripts: {FIGURE_SCRIPT_DIR}/")


## Load the frozen hNDE event model

The PRESEL classifier, post-selection yields, reference flow, and the two four-member density-ratio ensembles are the same objects used by Exercise 5. No event-level model is retrained here.


In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_CANDIDATES = [
    CACHE_DIR / "exercise5_preselection_state.npz",
    Path("saved_asimov_nis_influence_v2/exercise5_preselection_state.npz"),
    Path("saved_exercise7_misspecification/exercise5_preselection_state.npz"),
]
existing_state = next(
    (path for path in PRESEL_STATE_CANDIDATES if path.exists()), None
)
if existing_state is not None:
    state = np.load(existing_state)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded PRESEL state from {existing_state}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0], PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms, statistics = {}, {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES,
                ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges,
                batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION,
                split_seed=SPLIT_SEED,
            )
        )
    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"], histograms["background"], edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_CANDIDATES[0],
        ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG,
        lambda_background=LAM_BKG,
    )

reference_flow = load_flow(
    "reference",
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE,
    device=device,
    expected_features=FEATURES,
)
ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )

print(f"PRESEL ratio cut: {PRESEL_RATIO_CUT:.6g}")
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")


## Reconstruct and cache the compressed Exercise 5 likelihood

For every event, the fitted likelihood depends only on

$$
q(x)=\frac{\lambda_S r_S(x)}{\lambda_B r_B(x)}.
$$

We draw the same five-million-event reference sample as Exercise 5, normalize both process ratios on it, and compress $\log q$ into 512 bins. The compressed signal/background probabilities generate hNDE toys; their ratio defines the **frozen likelihood** used to fit both hNDE and simulator toys.


In [ ]:
def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(
            values[start : start + int(batch_size)], columns=FEATURES
        )
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch, scaler=pack["scaler"], model=pack["model"]
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks) if chunks else np.empty(0)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"Non-finite {sample_name} ratio.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


COMPRESSED_MODEL_PATH = CACHE_DIR / "compressed_exercise5_model.npz"
compression_cache_metadata = {
    "cache_version": 1,
    "n_reference_events": N_REFERENCE_EVENTS,
    "toy_q_bins": TOY_Q_BINS,
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
    "presel_ratio_cut": PRESEL_RATIO_CUT,
}
if COMPRESSED_MODEL_PATH.exists():
    saved = np.load(COMPRESSED_MODEL_PATH)
    missing_metadata = set(compression_cache_metadata) - set(saved.files)
    mismatched_metadata = [
        name for name, expected in compression_cache_metadata.items()
        if name in saved.files
        and not np.isclose(float(saved[name]), float(expected), rtol=1e-12)
    ]
    if missing_metadata or mismatched_metadata:
        raise RuntimeError(
            "The compressed-model cache has stale provenance. Bump "
            "BASE_RUN_TAG (recommended) or remove only that versioned cache. "
            f"Missing={sorted(missing_metadata)}, "
            f"mismatched={mismatched_metadata}."
        )
    COMPRESSED_Q = saved["q"]
    HNDE_SIGNAL_PROBABILITY = saved["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = saved["background_probability"]
    LOG_Q_EDGES = saved["log_q_edges"]
    RATIO_NORMALIZATION = {
        "signal": float(saved["signal_normalization"]),
        "background": float(saved["background_normalization"]),
    }
    compression_truth = saved["validation_truth"]
    compression_test = saved["validation_test"]
    compression_unbinned = saved["validation_unbinned"]
    compression_binned = saved["validation_binned"]
    saved.close()
    print(f"Loaded compressed hNDE model from {COMPRESSED_MODEL_PATH}")
else:
    torch.manual_seed(SEED + 100)
    reference_values, reference_acceptance = sample_preselected_flow(
        reference_flow, N_REFERENCE_EVENTS, REFERENCE_SAMPLING_BATCH_SIZE
    )
    raw_signal = evaluate_ratio("signal", reference_values)
    raw_background = evaluate_ratio("background", reference_values)
    RATIO_NORMALIZATION = {
        "signal": float(raw_signal.mean()),
        "background": float(raw_background.mean()),
    }
    ratio_signal = raw_signal / RATIO_NORMALIZATION["signal"]
    ratio_background = raw_background / RATIO_NORMALIZATION["background"]
    weight_signal = ratio_signal / N_REFERENCE_EVENTS
    weight_background = ratio_background / N_REFERENCE_EVENTS
    event_q = (
        LAM_SIG / LAM_BKG * ratio_signal / ratio_background
    )
    compressed = build_compressed_q_model(
        event_q,
        weight_signal,
        weight_background,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        n_bins=TOY_Q_BINS,
    )
    COMPRESSED_Q = compressed["q"]
    HNDE_SIGNAL_PROBABILITY = compressed["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = compressed["background_probability"]
    LOG_Q_EDGES = compressed["log_q_edges"]

    compression_rows = []
    for truth_mu in [0.0, 0.25, 1.0, 2.0, 3.0]:
        unbinned_expected = (
            truth_mu * LAM_SIG * weight_signal
            + LAM_BKG * weight_background
        )
        binned_expected = (
            truth_mu * LAM_SIG * HNDE_SIGNAL_PROBABILITY
            + LAM_BKG * HNDE_BACKGROUND_PROBABILITY
        )
        for test_mu in [0.0, 0.5, 1.0, 2.0, 3.0]:
            if test_mu == truth_mu:
                continue
            compression_rows.append((
                truth_mu,
                test_mu,
                asimov_test_statistic(
                    test_mu, truth_mu, event_q, unbinned_expected,
                    lam_signal=LAM_SIG,
                ),
                asimov_test_statistic(
                    test_mu, truth_mu, COMPRESSED_Q, binned_expected,
                    lam_signal=LAM_SIG,
                ),
            ))
    compression_rows = np.asarray(compression_rows, dtype=np.float64)
    compression_truth = compression_rows[:, 0]
    compression_test = compression_rows[:, 1]
    compression_unbinned = compression_rows[:, 2]
    compression_binned = compression_rows[:, 3]
    np.savez_compressed(
        COMPRESSED_MODEL_PATH,
        q=COMPRESSED_Q,
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        log_q_edges=LOG_Q_EDGES,
        signal_normalization=RATIO_NORMALIZATION["signal"],
        background_normalization=RATIO_NORMALIZATION["background"],
        validation_truth=compression_truth,
        validation_test=compression_test,
        validation_unbinned=compression_unbinned,
        validation_binned=compression_binned,
        **compression_cache_metadata,
    )
    print(f"Saved compressed hNDE model to {COMPRESSED_MODEL_PATH}")
    print(f"Reference PRESEL acceptance: {reference_acceptance:.3%}")
    del reference_values, raw_signal, raw_background
    del ratio_signal, ratio_background, weight_signal, weight_background, event_q
    gc.collect()

print("Ratio normalizations:", RATIO_NORMALIZATION)
print("Compressed q quantiles:", np.quantile(COMPRESSED_Q, [0, .01, .5, .99, 1]))



# --------------------------------------------------------------------------
# Reconstruct Exercise 5's retained evaluation reservoir and seal its unused
# event-level complement before releasing the frozen PRESEL/ratio networks.
# --------------------------------------------------------------------------
saved_simulator_templates = simulator_templates_from_exercise5(
    weights_path=HYBRID_DENSITY_DIR / "weights_asimov.npy",
    ratio_signal_path=HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    ratio_background_path=HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
    log_q_edges=LOG_Q_EDGES,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    seed=SEED + 600,
    source_ratio_normalization=EXERCISE5_RATIO_NORMALIZATION,
    target_ratio_normalization=RATIO_NORMALIZATION,
)


TEMPLATE_CACHE_VERSION = 3


def update_array_digest(digest, label, values):
    values = np.ascontiguousarray(np.asarray(values))
    digest.update(str(label).encode("utf-8"))
    digest.update(str(values.dtype).encode("utf-8"))
    digest.update(str(values.shape).encode("utf-8"))
    digest.update(values.view(np.uint8))


construction_source_digest = hashlib.sha256()
for source_name in (
    "pooled_signal_probability",
    "pooled_background_probability",
    "pooled_signal_counts",
    "pooled_background_counts",
):
    update_array_digest(
        construction_source_digest,
        source_name,
        saved_simulator_templates[source_name],
    )
CONSTRUCTION_SOURCE_FINGERPRINT = (
    construction_source_digest.hexdigest()
)
EXPECTED_TEMPLATE_EVENTS = {
    "construction": {
        process: EXERCISE5_EVAL_CAP
        for process in ("signal", "background")
    },
    "audit": {
        process: (
            EXERCISE5_EXPECTED_SELECTED_EVAL[process]
            - EXERCISE5_EVAL_CAP
        )
        for process in ("signal", "background")
    },
}
TEMPLATE_CACHE_METADATA = {
    "template_cache_version": TEMPLATE_CACHE_VERSION,
    "exercise5_eval_cap": EXERCISE5_EVAL_CAP,
    "signal_eval_priority_seed": (
        EXERCISE5_EVAL_PRIORITY_SEED["signal"]
    ),
    "background_eval_priority_seed": (
        EXERCISE5_EVAL_PRIORITY_SEED["background"]
    ),
    "signal_expected_selected_eval": (
        EXERCISE5_EXPECTED_SELECTED_EVAL["signal"]
    ),
    "background_expected_selected_eval": (
        EXERCISE5_EXPECTED_SELECTED_EVAL["background"]
    ),
    "split_seed": SPLIT_SEED,
    "presel_train_fraction": PRESEL_TRAIN_FRACTION,
    "flow_train_fraction": FLOW_TRAIN_FRACTION,
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
    "exercise5_source_signal_normalization": (
        EXERCISE5_RATIO_NORMALIZATION["signal"]
    ),
    "exercise5_source_background_normalization": (
        EXERCISE5_RATIO_NORMALIZATION["background"]
    ),
    "construction_source_fingerprint": (
        CONSTRUCTION_SOURCE_FINGERPRINT
    ),
}


def compressed_simulator_template(rows):
    values = rows[FEATURES].to_numpy(dtype=np.float32)
    ratio_signal = evaluate_ratio("signal", values) / RATIO_NORMALIZATION[
        "signal"
    ]
    ratio_background = evaluate_ratio(
        "background", values
    ) / RATIO_NORMALIZATION["background"]
    log_q = (
        np.log(LAM_SIG / LAM_BKG)
        + np.log(ratio_signal)
        - np.log(ratio_background)
    )
    counts = np.histogram(log_q, bins=LOG_Q_EDGES)[0].astype(np.int64)
    probability = np.histogram(
        log_q,
        bins=LOG_Q_EDGES,
        weights=rows["weight"].to_numpy(dtype=np.float64),
    )[0].astype(np.float64)
    probability += 1.0e-15
    probability /= probability.sum()
    return probability, counts


def validated_template_pair_fingerprint(
    path, expected_events, expected_counts=None
):
    """Validate a cached template without exposing its law."""
    if not path.exists():
        return None
    required = {
        "log_q_edges", "presel_ratio_cut",
        "signal_ratio_normalization",
        "background_ratio_normalization",
        "signal_probability", "background_probability",
        "signal_counts", "background_counts",
        "signal_events", "background_events",
        "reservoir_pair_fingerprint",
        *TEMPLATE_CACHE_METADATA.keys(),
    }
    try:
        with np.load(path) as saved:
            if not required.issubset(saved.files):
                return None
            if not np.array_equal(saved["log_q_edges"], LOG_Q_EDGES):
                return None
            if not np.isclose(
                float(saved["presel_ratio_cut"]), PRESEL_RATIO_CUT,
                rtol=0.0, atol=1.0e-12,
            ):
                return None
            for process in ("signal", "background"):
                if not np.isclose(
                    float(saved[f"{process}_ratio_normalization"]),
                    RATIO_NORMALIZATION[process],
                    rtol=0.0, atol=1.0e-12,
                ):
                    return None
            for name, expected in TEMPLATE_CACHE_METADATA.items():
                observed = saved[name].item()
                if isinstance(expected, str):
                    if str(observed) != expected:
                        return None
                elif not np.isclose(
                    float(observed), float(expected),
                    rtol=0.0, atol=1.0e-12,
                ):
                    return None

            n_bins = len(LOG_Q_EDGES) - 1
            for process in ("signal", "background"):
                probability = np.asarray(
                    saved[f"{process}_probability"],
                    dtype=np.float64,
                )
                raw_counts = np.asarray(saved[f"{process}_counts"])
                counts = raw_counts.astype(np.int64)
                events = int(saved[f"{process}_events"])
                if (
                    probability.shape != (n_bins,)
                    or counts.shape != (n_bins,)
                    or not np.array_equal(raw_counts, counts)
                    or not np.isfinite(probability).all()
                    or np.any(probability < 0.0)
                    or not np.isclose(probability.sum(), 1.0)
                    or np.any(counts < 0)
                    or counts.sum() != events
                    or events != expected_events[process]
                ):
                    return None
                if (
                    expected_counts is not None
                    and not np.array_equal(
                        counts, expected_counts[process]
                    )
                ):
                    return None
            pair_fingerprint = str(
                saved["reservoir_pair_fingerprint"].item()
            )
            if (
                len(pair_fingerprint) != 64
                or any(
                    character not in "0123456789abcdef"
                    for character in pair_fingerprint
                )
            ):
                return None
            return pair_fingerprint
    except (OSError, ValueError, KeyError, TypeError):
        return None


canonical_template_counts = {
    process: saved_simulator_templates[f"pooled_{process}_counts"]
    for process in ("signal", "background")
}
construction_pair_fingerprint = (
    validated_template_pair_fingerprint(
        CONSTRUCTION_TEMPLATE_PATH,
        EXPECTED_TEMPLATE_EVENTS["construction"],
        canonical_template_counts,
    )
)
audit_pair_fingerprint = validated_template_pair_fingerprint(
    SEALED_AUDIT_TEMPLATE_PATH,
    EXPECTED_TEMPLATE_EVENTS["audit"],
)
templates_ready = (
    construction_pair_fingerprint is not None
    and construction_pair_fingerprint == audit_pair_fingerprint
)
if not templates_ready and (
    CONSTRUCTION_TEMPLATE_PATH.exists()
    or SEALED_AUDIT_TEMPLATE_PATH.exists()
):
    print("Cached simulator-reservoir pair is stale; rebuilding it.")
if not templates_ready:
    construction_payload = {
        name: np.asarray(value)
        for name, value in TEMPLATE_CACHE_METADATA.items()
    }
    construction_payload.update({
        "log_q_edges": LOG_Q_EDGES,
        "presel_ratio_cut": np.asarray(PRESEL_RATIO_CUT),
        "signal_ratio_normalization": np.asarray(
            RATIO_NORMALIZATION["signal"]
        ),
        "background_ratio_normalization": np.asarray(
            RATIO_NORMALIZATION["background"]
        ),
    })
    audit_payload = dict(construction_payload)
    reservoir_pair_digest = hashlib.sha256()
    reservoir_pair_digest.update(
        CONSTRUCTION_SOURCE_FINGERPRINT.encode("utf-8")
    )
    for process in ("signal", "background"):
        print(f"Reconstructing the complete selected eval split: {process}")
        selected_eval, eval_stats = collect_preselected_eval_rows(
            SAMPLE_PATHS[process],
            features=FEATURES,
            ratio_predictor=evaluate_PRESEL_ratio,
            ratio_cut=PRESEL_RATIO_CUT,
            batch_size=STREAM_BATCH_SIZE,
            presel_fraction=PRESEL_TRAIN_FRACTION,
            flow_train_fraction=FLOW_TRAIN_FRACTION,
            split_seed=SPLIT_SEED,
        )
        expected_selected = EXERCISE5_EXPECTED_SELECTED_EVAL[process]
        if len(selected_eval) != expected_selected:
            raise RuntimeError(
                f"The {process} eval split has {len(selected_eval):,} "
                f"selected rows, expected {expected_selected:,}. The raw "
                "events or frozen PRESEL state differ from Exercise 5, so "
                "the claimed unused audit complement cannot be certified."
            )

        priority = deterministic_row_priority(
            selected_eval["_row_index"].to_numpy(dtype=np.uint64),
            EXERCISE5_EVAL_PRIORITY_SEED[process],
        )
        order = np.argsort(priority, kind="stable")
        retained = selected_eval.iloc[order[:EXERCISE5_EVAL_CAP]].copy()
        unused = selected_eval.iloc[order[EXERCISE5_EVAL_CAP:]].copy()
        if len(retained) != EXERCISE5_EVAL_CAP or len(unused) == 0:
            raise RuntimeError("The deterministic eval-reservoir split failed.")
        if np.intersect1d(
            retained["_row_index"].to_numpy(dtype=np.uint64),
            unused["_row_index"].to_numpy(dtype=np.uint64),
        ).size:
            raise RuntimeError("Construction and audit event reservoirs overlap.")
        for subset_name, subset_rows in (
            ("retained", retained), ("unused", unused)
        ):
            update_array_digest(
                reservoir_pair_digest,
                f"{process}:{subset_name}:row_indices",
                subset_rows["_row_index"].to_numpy(
                    dtype=np.uint64
                ),
            )

        reconstructed_probability, reconstructed_counts = (
            compressed_simulator_template(retained)
        )
        canonical_probability = saved_simulator_templates[
            f"pooled_{process}_probability"
        ]
        reconstruction_l1 = float(np.sum(np.abs(
            reconstructed_probability - canonical_probability
        )))
        print(
            f"  reconstructed retained rows={len(retained):,}; "
            f"unused sealed rows={len(unused):,}; "
            f"saved-template L1={reconstruction_l1:.3e}"
        )
        if reconstruction_l1 > 1.0e-3:
            raise RuntimeError(
                f"The {process} Exercise 5 reservoir was not reproduced "
                "closely enough. The unused complement is not certified."
            )
        canonical_counts = saved_simulator_templates[
            f"pooled_{process}_counts"
        ]
        count_migration_fraction = float(
            0.5 * np.sum(np.abs(
                reconstructed_counts - canonical_counts
            )) / canonical_counts.sum()
        )
        print(
            f"  compressed-bin migration fraction="
            f"{count_migration_fraction:.3e}"
        )
        if count_migration_fraction > 1.0e-4:
            raise RuntimeError(
                f"The reconstructed {process} compressed-bin counts "
                "differ too much from the saved Exercise 5 reservoir."
            )

        audit_probability, audit_counts = compressed_simulator_template(unused)
        construction_payload[f"{process}_probability"] = canonical_probability
        construction_payload[f"{process}_counts"] = canonical_counts
        construction_payload[f"{process}_events"] = np.asarray(len(retained))
        audit_payload[f"{process}_probability"] = audit_probability
        audit_payload[f"{process}_counts"] = audit_counts
        audit_payload[f"{process}_events"] = np.asarray(len(unused))
        audit_payload[f"{process}_row_index_min"] = np.asarray(
            unused["_row_index"].min(), dtype=np.uint64
        )
        audit_payload[f"{process}_row_index_max"] = np.asarray(
            unused["_row_index"].max(), dtype=np.uint64
        )
        del selected_eval, retained, unused, priority, eval_stats
        gc.collect()

    reservoir_pair_fingerprint = reservoir_pair_digest.hexdigest()
    construction_payload["reservoir_pair_fingerprint"] = np.asarray(
        reservoir_pair_fingerprint
    )
    audit_payload["reservoir_pair_fingerprint"] = np.asarray(
        reservoir_pair_fingerprint
    )
    construction_temporary = CONSTRUCTION_TEMPLATE_PATH.with_suffix(".tmp.npz")
    audit_temporary = SEALED_AUDIT_TEMPLATE_PATH.with_suffix(".tmp.npz")
    np.savez_compressed(construction_temporary, **construction_payload)
    np.savez_compressed(audit_temporary, **audit_payload)
    construction_temporary.replace(CONSTRUCTION_TEMPLATE_PATH)
    audit_temporary.replace(SEALED_AUDIT_TEMPLATE_PATH)
    construction_pair_fingerprint = (
        validated_template_pair_fingerprint(
            CONSTRUCTION_TEMPLATE_PATH,
            EXPECTED_TEMPLATE_EVENTS["construction"],
            canonical_template_counts,
        )
    )
    audit_pair_fingerprint = validated_template_pair_fingerprint(
        SEALED_AUDIT_TEMPLATE_PATH,
        EXPECTED_TEMPLATE_EVENTS["audit"],
    )
    if (
        construction_pair_fingerprint is None
        or construction_pair_fingerprint != audit_pair_fingerprint
    ):
        raise RuntimeError(
            "The newly saved simulator-reservoir pair failed "
            "its provenance validation."
        )
    print("Saved construction template:", CONSTRUCTION_TEMPLATE_PATH)
    print("Sealed unused-event audit template:", SEALED_AUDIT_TEMPLATE_PATH)
    del (
        audit_payload, construction_payload, audit_probability,
        audit_counts, canonical_probability, canonical_counts,
        reconstructed_probability, reconstructed_counts, order,
    )
    gc.collect()
else:
    print("Loaded certified simulator-reservoir provenance from cache.")

with np.load(CONSTRUCTION_TEMPLATE_PATH) as construction_template:
    RESERVOIR_PAIR_FINGERPRINT = str(
        construction_template["reservoir_pair_fingerprint"].item()
    )
    SIM_CALIBRATION_SIGNAL_PROBABILITY = construction_template[
        "signal_probability"
    ].astype(np.float64)
    SIM_CALIBRATION_BACKGROUND_PROBABILITY = construction_template[
        "background_probability"
    ].astype(np.float64)
    SIM_CALIBRATION_SIGNAL_COUNTS = construction_template[
        "signal_counts"
    ].astype(np.int64)
    SIM_CALIBRATION_BACKGROUND_COUNTS = construction_template[
        "background_counts"
    ].astype(np.int64)
    SIM_CALIBRATION_EVENTS = {
        "signal": int(construction_template["signal_events"]),
        "background": int(construction_template["background_events"]),
    }
if SIM_CALIBRATION_EVENTS != EXPECTED_TEMPLATE_EVENTS["construction"]:
    raise RuntimeError(
        "The construction template has stale event counts."
    )
for probability in (
    SIM_CALIBRATION_SIGNAL_PROBABILITY,
    SIM_CALIBRATION_BACKGROUND_PROBABILITY,
):
    if not np.isclose(probability.sum(), 1.0):
        raise RuntimeError("A construction simulator template is not normalized.")
print("Construction simulator events:", SIM_CALIBRATION_EVENTS)
print(
    "The audit cache passed provenance-only validation; its law remains "
    "unavailable to the construction until the final audit cell."
)

# The frozen event networks are no longer needed after compression. Free
# their Torch/ONNX GPU allocations before the large statistic flow trains.
reference_flow = None
ratio_models = None
PRESEL_model = None
PRESEL_scaler = None
model_proto = None
scaler = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released frozen Exercise 5 event networks.")


### Validate the compression over the full design interval

Exercise 5 checked one Asimov displacement. Here the unbinned and compressed expected statistics are compared for several generating and tested values across $[0,3]$. This validates the numerical acceleration before it is used half a million times.


In [ ]:
compression_validation = pd.DataFrame({
    "mu_true": compression_truth,
    "mu_test": compression_test,
    "t_unbinned": compression_unbinned,
    "t_compressed": compression_binned,
})
compression_validation["absolute_difference"] = (
    compression_validation["t_compressed"]
    - compression_validation["t_unbinned"]
)
compression_validation["relative_difference"] = np.divide(
    compression_validation["absolute_difference"],
    compression_validation["t_unbinned"],
    out=np.zeros(len(compression_validation)),
    where=compression_validation["t_unbinned"] > 1.0e-8,
)
display(compression_validation.style.format(precision=6).hide(axis="index"))
max_relative = float(compression_validation["relative_difference"].abs().max())
max_absolute = float(compression_validation["absolute_difference"].abs().max())
print(f"Maximum relative/absolute change: {max_relative:.3%} / {max_absolute:.4g}")
if max_relative > 5.0e-3 and max_absolute > 2.0e-2:
    raise RuntimeError(
        "The 512-bin compression is not accurate to 0.5% across the scan. "
        "Increase TOY_Q_BINS."
    )


## Vectorized pseudo-experiment fits

For compressed counts $n_j$, the hNDE-coordinate MLE $\widehat\nu$ solves

$$
0=\lambda_S-\sum_j n_j\frac{q_j}{1+\widehat\nu q_j},
\qquad \widehat\nu\geq0.
$$

JAX performs 16 bounded Newton steps for an entire toy batch, with a vectorized bisection fallback. For any tested hNDE coordinate $\nu_0$ it returns

$$
t(\nu_0)=2\left[(\nu_0-\widehat\nu)\lambda_S
-\sum_j n_j\left\{\log(1+\nu_0q_j)
-\log(1+\widehat\nu q_j)\right\}\right].
$$

The response-corrected statistic is obtained by passing $\nu_0=g(\mu)$. The toy generator separately records the physical parameter $\mu$, the parameter used to generate counts, and the tested likelihood coordinate. This separation is essential: hNDE toys use $g(\mu)$ for both generation and testing, whereas simulator toys generate at physical $\mu$ and test $g(\mu)$.


In [ ]:
COMPRESSED_Q_JAX = jnp.asarray(COMPRESSED_Q)


@jax.jit
def fit_compressed_toy_batch(counts, test_mu):
    counts = jnp.asarray(counts, dtype=jnp.float64)
    test_mu = jnp.asarray(test_mu, dtype=jnp.float64).reshape(-1)
    q_values = COMPRESSED_Q_JAX
    initial_mu = jnp.clip(
        (jnp.sum(counts, axis=1) - LAM_BKG) / LAM_SIG,
        0.0,
        TOY_MU_MAX,
    )
    score_at_zero = LAM_SIG - jnp.sum(counts * q_values, axis=1)

    def newton_step(_, mu):
        response = q_values / (1.0 + mu[:, None] * q_values)
        score = LAM_SIG - jnp.sum(counts * response, axis=1)
        information = jnp.sum(counts * response**2, axis=1)
        step = jnp.clip(
            score / jnp.maximum(information, 1.0e-12), -2.0, 2.0
        )
        return jnp.clip(mu - step, 0.0, TOY_MU_MAX)

    mu_hat = jax.lax.fori_loop(
        0, TOY_NEWTON_STEPS, newton_step, initial_mu
    )
    mu_hat = jnp.where(score_at_zero >= 0.0, 0.0, mu_hat)
    statistic = 2.0 * (
        (test_mu - mu_hat) * LAM_SIG
        - jnp.sum(
            counts * (
                jnp.log1p(test_mu[:, None] * q_values)
                - jnp.log1p(mu_hat[:, None] * q_values)
            ),
            axis=1,
        )
    )
    fitted_response = q_values / (1.0 + mu_hat[:, None] * q_values)
    fitted_score = LAM_SIG - jnp.sum(
        counts * fitted_response, axis=1
    )
    fitted_score = jnp.where(mu_hat == 0.0, 0.0, fitted_score)
    return mu_hat, jnp.maximum(statistic, 0.0), fitted_score


def fit_toy_batch_numpy(counts, test_mu):
    counts = np.asarray(counts)
    test_mu = np.asarray(test_mu, dtype=np.float64).reshape(-1)
    result = fit_compressed_toy_batch(counts, test_mu)
    mu_hat, t_mu, fitted_score = [
        np.asarray(values, dtype=np.float64) for values in result
    ]
    failed = (
        (mu_hat > 1.0e-10)
        & (mu_hat < TOY_MU_MAX - 1.0e-10)
        & (np.abs(fitted_score) > 1.0e-6)
    )
    if np.any(failed):
        # The score is monotone increasing in mu. A vectorized bisection
        # fallback makes rare Newton failures harmless without returning
        # to one-Minuit-fit-per-toy execution.
        failed_counts = counts[failed].astype(np.float64)
        lower = np.zeros(np.sum(failed), dtype=np.float64)
        upper = np.full(np.sum(failed), TOY_MU_MAX, dtype=np.float64)
        for _ in range(64):
            middle = 0.5 * (lower + upper)
            response = COMPRESSED_Q / (
                1.0 + middle[:, None] * COMPRESSED_Q
            )
            middle_score = LAM_SIG - np.sum(
                failed_counts * response, axis=1
            )
            move_lower = middle_score < 0.0
            lower = np.where(move_lower, middle, lower)
            upper = np.where(move_lower, upper, middle)
        repaired_mu = 0.5 * (lower + upper)
        mu_hat[failed] = repaired_mu
        failed_test_mu = test_mu[failed]
        repaired_t = 2.0 * (
            (failed_test_mu - repaired_mu) * LAM_SIG
            - np.sum(
                failed_counts
                * (
                    np.log1p(failed_test_mu[:, None] * COMPRESSED_Q)
                    - np.log1p(repaired_mu[:, None] * COMPRESSED_Q)
                ),
                axis=1,
            )
        )
        t_mu[failed] = np.maximum(repaired_t, 0.0)
        repaired_response = COMPRESSED_Q / (
            1.0 + repaired_mu[:, None] * COMPRESSED_Q
        )
        fitted_score[failed] = LAM_SIG - np.sum(
            failed_counts * repaired_response, axis=1
        )
    if np.any(mu_hat >= TOY_MU_MAX - 1.0e-10):
        raise RuntimeError(
            "A toy MLE reached TOY_MU_MAX; increase the fit bound."
        )
    return mu_hat, t_mu, fitted_score


# Compile once and check that the fitted score is small away from the boundary.
_rng = np.random.default_rng(SEED + 200)
_mu = _rng.uniform(*MU_RANGE, size=8)
_mean = (
    _mu[:, None] * LAM_SIG * HNDE_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * HNDE_BACKGROUND_PROBABILITY[None, :]
)
_counts = _rng.poisson(_mean)
_muhat, _tmu, _score = fit_toy_batch_numpy(_counts, _mu)
print("Toy-kernel smoke test:")
print("  mu_true:", np.round(_mu, 3))
print("  mu_hat: ", np.round(_muhat, 3))
print("  t_mu:  ", np.round(_tmu, 3))
print(f"  max interior score residual: {np.max(np.abs(_score)):.3e}")
del _rng, _mu, _mean, _counts, _muhat, _tmu, _score


## 1. Learn the monotone pseudo-truth response $g(\mu)$

Regressing the finite-sample mean of $\widehat\nu$ would mix the population response with estimator bias and the $\widehat\nu\geq0$ boundary. Instead we compute the KL pseudo-truth directly from the construction simulator template. For compressed simulator means

$$
m_j^{\rm sim}(\mu)
=\mu\lambda_S P^{\rm sim}_{S,j}
+\lambda_B P^{\rm sim}_{B,j},
$$

$g(\mu)$ obeys the population score equation

$$
\lambda_S-\sum_j m_j^{\rm sim}(\mu)
\frac{q_j}{1+g(\mu)q_j}=0,
$$

unless the constrained optimum is $g(\mu)=0$. Positivity of the templates and of $q_j$ makes this response nondecreasing. We solve it on a dense physical-$\mu$ grid, validate the KKT condition, interpolate with a monotone PCHIP, and check the interpolation at every midpoint.

Crucially, the construction template alone defines $g$. The unused audit-event template remains sealed. We do not force $g(0)=0$ or $g(3)=3$: either constraint would erase precisely the simulator mismatch we are trying to represent.


In [ ]:
RESPONSE_MU_GRID = np.linspace(*MU_RANGE, RESPONSE_GRID_POINTS)
response_expected_counts = (
    RESPONSE_MU_GRID[:, None]
    * LAM_SIG
    * SIM_CALIBRATION_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * SIM_CALIBRATION_BACKGROUND_PROBABILITY[None, :]
)
response_grid, _, _ = fit_toy_batch_numpy(
    response_expected_counts, RESPONSE_MU_GRID
)
response_grid = np.asarray(response_grid, dtype=np.float64)
response_score = LAM_SIG - np.sum(
    response_expected_counts
    * COMPRESSED_Q[None, :]
    / (1.0 + response_grid[:, None] * COMPRESSED_Q[None, :]),
    axis=1,
)
response_interior = response_grid > 1.0e-10
response_kkt_violation = np.where(
    response_interior,
    np.abs(response_score),
    np.maximum(-response_score, 0.0),
)
if np.max(response_kkt_violation) > 1.0e-6:
    raise RuntimeError(
        "The pseudo-truth response does not satisfy the constrained "
        "population score equation."
    )
if np.min(np.diff(response_grid)) < -1.0e-10:
    raise RuntimeError("The fitted pseudo-truth response is not monotone.")
if response_grid.min() < 0.0 or response_grid.max() >= TOY_MU_MAX - 0.5:
    raise RuntimeError(
        "The pseudo-truth response lies outside the safe hNDE fit range."
    )

response_interpolator = PchipInterpolator(
    RESPONSE_MU_GRID, response_grid, extrapolate=False
)


def pseudo_truth_response(mu):
    """Map physical simulator truth to the frozen hNDE coordinate."""
    values = np.asarray(mu, dtype=np.float64)
    tolerance = 1.0e-12
    if (
        not np.isfinite(values).all()
        or np.any(values < MU_RANGE[0] - tolerance)
        or np.any(values > MU_RANGE[1] + tolerance)
    ):
        raise ValueError("Physical mu lies outside the response-map range.")
    result = np.asarray(
        response_interpolator(np.clip(values, *MU_RANGE)),
        dtype=np.float64,
    )
    return result


response_midpoints = 0.5 * (
    RESPONSE_MU_GRID[:-1] + RESPONSE_MU_GRID[1:]
)
midpoint_expected_counts = (
    response_midpoints[:, None]
    * LAM_SIG
    * SIM_CALIBRATION_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * SIM_CALIBRATION_BACKGROUND_PROBABILITY[None, :]
)
midpoint_exact, _, _ = fit_toy_batch_numpy(
    midpoint_expected_counts, response_midpoints
)
response_midpoint_error = float(np.max(np.abs(
    pseudo_truth_response(response_midpoints) - midpoint_exact
)))
if response_midpoint_error > 5.0e-5:
    raise RuntimeError(
        "The response-map interpolation is not accurate enough; increase "
        "RESPONSE_GRID_POINTS."
    )

response_digest = hashlib.sha256()
for values in (RESPONSE_MU_GRID, response_grid):
    response_digest.update(np.ascontiguousarray(values).view(np.uint8))
RESPONSE_MAP_FINGERPRINT = (
    "simulator_pseudotruth_v2_" + response_digest.hexdigest()[:16]
)


def response_corrected_test_statistic(counts, physical_mu):
    """Fit counts and evaluate t_mu^(g) at physical ``mu``."""
    return fit_toy_batch_numpy(
        counts, pseudo_truth_response(physical_mu)
    )


print(
    "Pseudo-truth response endpoints:",
    f"g(0)={response_grid[0]:.6f}, g(3)={response_grid[-1]:.6f}",
)
print(
    "Response displacement range:",
    f"[{np.min(response_grid - RESPONSE_MU_GRID):+.6f}, "
    f"{np.max(response_grid - RESPONSE_MU_GRID):+.6f}]",
)
print("Maximum response interpolation error:", response_midpoint_error)
print("Response-map fingerprint:", RESPONSE_MAP_FINGERPRINT)

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.plot(RESPONSE_MU_GRID, response_grid, lw=2.4, label=r"$g(\mu)$")
ax.plot(MU_RANGE, MU_RANGE, "k--", lw=1.2, label="identity")
ax.set(
    xlim=MU_RANGE,
    xlabel=r"physical simulator truth $\mu$",
    ylabel=r"pseudo-true hNDE coordinate $g(\mu)$",
    title="Monotone simulator-to-hNDE response",
)
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
export_exercise11_figure(fig, "pseudo_truth_response_map")
plt.show()


## 2. Generate 500,000 response-matched hNDE toys

Draw physical design points $\mu\sim U(0,3)$, map them to $g(\mu)$, generate the inexpensive hNDE toy at that surrogate coordinate, and evaluate the numerator at the same coordinate. Thus these toys learn the central response-matched reference law

$$
p_{\rm H}^{(g)}(t\mid\mu),
\qquad
\mathcal D_{\rm H}\sim p_{\rm H}(\cdot\mid g(\mu)),
\qquad
t=t_\mu^{(g)}(\mathcal D_{\rm H}).
$$

The context stored for every flow and classifier remains the physical $\mu$, not $g(\mu)$. The cache records all three coordinates and aborts if a stale identity-response shard is encountered.


In [ ]:
hnde_toys = run_cached_toy_ensemble(
    cache_dir=CACHE_DIR / f"hnde_uniform_{N_HNDE_TOYS}",
    n_toys=N_HNDE_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 300,
    mu_range=MU_RANGE,
    signal_probability=HNDE_SIGNAL_PROBABILITY,
    background_probability=HNDE_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    generation_mu_transform=pseudo_truth_response,
    test_mu_transform=pseudo_truth_response,
    generation_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)
expected_hnde_coordinate = pseudo_truth_response(hnde_toys["mu"])
if not np.allclose(
    hnde_toys["generation_mu"], expected_hnde_coordinate,
    rtol=0.0, atol=2.0e-6,
) or not np.allclose(
    hnde_toys["test_mu"], expected_hnde_coordinate,
    rtol=0.0, atol=2.0e-6,
):
    raise RuntimeError("The response-matched hNDE cache has stale coordinates.")
print(pd.DataFrame({
    "mu": hnde_toys["mu"],
    "g_mu": hnde_toys["test_mu"],
    "mu_hat": hnde_toys["mu_hat"],
    "t_mu": hnde_toys["t_mu"],
    "n_events": hnde_toys["n_events"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())
print(
    "Maximum fitted-score residual:",
    f"{np.max(np.abs(hnde_toys['fitted_score'])):.3e}",
)

split_rng = np.random.default_rng(SEED + 301)
toy_order = split_rng.permutation(N_HNDE_TOYS)
flow_indices = toy_order[:N_FLOW_TOYS]
ratio1_indices = toy_order[N_FLOW_TOYS:]
flow_mu = hnde_toys["mu"][flow_indices].astype(np.float32)
flow_y = np.log(
    hnde_toys["t_mu"][flow_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)
ratio1_mu = hnde_toys["mu"][ratio1_indices].astype(np.float32)
ratio1_y_positive = np.log(
    hnde_toys["t_mu"][ratio1_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].hexbin(
    hnde_toys["mu"], hnde_toys["mu_hat"],
    gridsize=80, bins="log", mincnt=1, cmap="viridis",
)
response_plot_grid = np.linspace(*MU_RANGE, 400)
axes[0].plot(
    response_plot_grid, pseudo_truth_response(response_plot_grid),
    "w--", lw=1.5, label=r"$g(\mu)$",
)
axes[0].set(
    xlabel=r"physical $\mu$", ylabel=r"$\widehat\nu$",
    title="Response-matched hNDE pseudo-experiments",
)
axes[0].legend()
for low, high, color in [(0.0,.25,"C0"),(.75,1.0,"C1"),(1.75,2.0,"C2"),(2.75,3.0,"C3")]:
    mask = (hnde_toys["mu"] >= low) & (hnde_toys["mu"] < high)
    values = hnde_toys["t_mu"][mask]
    edges = np.linspace(0, np.quantile(values, .995), 60)
    axes[1].hist(values, bins=edges, density=True, histtype="step", lw=1.8,
                 color=color, label=rf"$\mu\in[{low:g},{high:g})$")
x = np.linspace(0.001, axes[1].get_xlim()[1], 400)
axes[1].plot(x, chi2.pdf(x, df=1), "k--", lw=1.3, label=r"$\chi^2_1$")
axes[1].set(
    xlabel=r"$t_{\mu}^{(g)}$", ylabel="Density",
    title="Response matching does not assume a pivot",
)
axes[1].legend(fontsize=8)
axes[1].set_yscale('log')
fig.tight_layout()
export_exercise11_figure(fig, "hnde_amortized_toys")
plt.show()


## 3. Train the conditional quadratic-spline reference

The first 400,000 response-matched hNDE toys train a conditional rational-quadratic-spline density in $y=\log(t_\mu^{(g)}+\epsilon)$. The remaining 100,000 toys are disjoint and are reserved for the first density-ratio correction.

The scalar flow is conditioned on physical $\mu$. Its target includes the non-Gaussian structure left after response matching; no Wilks or Wald approximation is imposed.


In [ ]:
statistic_flow = train_spline_flow(
    flow_y[:, None],
    context=flow_mu[:, None],
    checkpoint=MODEL_DIR / "q_phi_y_given_mu.pt",
    model_config=STATISTIC_FLOW_MODEL_CONFIG,
    training_config=STATISTIC_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 400,
    load_if_available=LOAD_IF_AVAILABLE,
)
flow_config_mismatch = {
    name: (statistic_flow["config"].get(name), expected)
    for name, expected in STATISTIC_FLOW_MODEL_CONFIG.items()
    if statistic_flow["config"].get(name) != expected
}
if flow_config_mismatch:
    raise RuntimeError(
        "The loaded statistic-flow checkpoint has stale architecture: "
        f"{flow_config_mismatch}. Bump BASE_RUN_TAG."
    )
print("Conditional statistic flow:", statistic_flow["checkpoint"])


## 4. First matched correction: spline $\rightarrow$ response-matched hNDE toys

At each physical $\mu_i$, a positive example is the held-out hNDE statistic $y_i$ and its matched negative example is drawn from the conditional spline at that same $\mu_i$. Ordinary BCE therefore estimates the residual density ratio without proposal-prior weights. Paired group splitting keeps the positive and negative members of each matched pair on the same side of the train/validation boundary.


In [ ]:
Y_MIN = float(np.log(T_OFFSET))
ratio1_y_negative, ratio1_rejection = sample_truncated_spline_flow(
    statistic_flow,
    ratio1_mu[:, None],
    lower_bound=Y_MIN,
    seed=SEED + 500,
)
ratio1_positive = np.column_stack([ratio1_mu, ratio1_y_positive])
ratio1_negative = np.column_stack([ratio1_mu, ratio1_y_negative])
paired_ids_1 = np.arange(N_RATIO1_TOYS, dtype=np.int64)
ratio1_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(f"Training hNDE residual member {member + 1}/{RATIO_ENSEMBLE_SIZE}")
    print("=" * 76)
    ratio1_ensemble.append(
        train_ratio_classifier(
            ratio1_positive,
            ratio1_negative,
            checkpoint=RATIO1_MODEL_DIR / f"r1_member{member}.pt",
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 510 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_ids_1,
        )
    )
print(f"Unphysical flow-reference rejection fraction: {ratio1_rejection:.4%}")
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in ratio1_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(ratio1_ensemble):
    history = pack.get("history", {})
    ax.plot(history.get("validation", []), label=f"member {member}")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="First conditional-ratio ensemble")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "first_ratio_training")
plt.show()


## 5. Normalize the response-matched hNDE density and compute $F_{\rm H}^{(g)}$

The first run exposed an important numerical trap. Increasing $Y_{\max}$ while holding the number of grid points fixed coarsened the grid, eventually producing an impossible raw-flow mass above one. The old loop also tested the density while its endpoint was still inside the learned spline region, where a rational-quadratic spline need not have a monotone tail.

The replacement below is self-adjusting:

1. it places $Y_{\max}$ beyond the standardized spline tail boundary, where every scalar `nflows` transform is exactly the identity and the raw-flow tail is an analytic standard-normal tail;
2. when support expands, it increases the point count to preserve the original $\Delta y$;
3. it doubles a nested grid until full/half-grid quantiles and integrals agree;
4. it compares numerical raw-flow mass with the exact normal-tail mass;
5. it uses the bounded classifier log-odds to place a conservative analytic upper bound on omitted hybrid tail mass.

All guards now change the relevant parameter before retrying and have finite hard caps. Failure raises an actionable exception; it is no longer possible to print a failed diagnostic and silently continue.


In [ ]:
MU_DENSITY_GRID = np.linspace(*MU_RANGE, QUADRATURE_MU_POINTS)
data_driven_y_max = float(max(
    5.0,
    np.quantile(np.concatenate([flow_y, ratio1_y_positive]), .99999) + 1.5,
))
target_y_mean = float(statistic_flow["target_scaler"].mean[0])
target_y_std = float(statistic_flow["target_scaler"].std[0])
spline_tail_bound = float(
    statistic_flow["config"]["spline_tail_bound"]
)

if target_y_std <= 0.0:
    raise RuntimeError("The statistic-flow target scale is not positive.")

Y_SUPPORT_BUFFER = 1.0
Y_UPPER_TAIL_MASS_TARGET = 1.0e-5
Y_QUANTILE_STABILITY_TARGET = 1.0e-2
Y_HALF_INTEGRAL_TARGET = 1.0e-3
Y_RAW_MASS_TARGET = 1.0e-3
Y_SUPPORT_MAX_EXPANSIONS = 3
Y_RESOLUTION_MAX_REFINEMENTS = 3
Y_GRID_MAX_POINTS = 32_769

# Preserve the original nominal spacing even if analytic tail certification
# requires a larger support. Use an even number of intervals so Y_GRID[::2]
# is a nested half-resolution grid with the same endpoints.
base_dy = (
    data_driven_y_max - Y_MIN
) / (QUADRATURE_Y_POINTS - 1)
Y_MAX = float(max(
    data_driven_y_max,
    target_y_mean + (spline_tail_bound + 2.0) * target_y_std,
))

quadrature_converged = False
for support_attempt in range(Y_SUPPORT_MAX_EXPANSIONS + 1):
    intervals = int(np.ceil((Y_MAX - Y_MIN) / base_dy))
    intervals += intervals % 2
    tail_requires_expansion = False

    for refinement in range(Y_RESOLUTION_MAX_REFINEMENTS + 1):
        n_points = intervals + 1
        if n_points > Y_GRID_MAX_POINTS:
            raise RuntimeError(
                f"Adaptive Y_GRID requires {n_points:,} points, above the "
                f"hard cap {Y_GRID_MAX_POINTS:,}. Increase the cap only "
                "after inspecting the statistic-flow boundary spike."
            )
        Y_GRID = np.linspace(Y_MIN, Y_MAX, n_points)
        flow_physical_grid = conditional_density_grid(
            statistic_flow,
            [],
            MU_DENSITY_GRID,
            Y_GRID,
        )
        hybrid1_grid = conditional_density_grid(
            statistic_flow,
            [ratio1_ensemble],
            MU_DENSITY_GRID,
            Y_GRID,
            max_abs_log_ratio=LOG_RATIO_CLIP,
        )

        z_min = (Y_MIN - target_y_mean) / target_y_std
        z_max = (Y_MAX - target_y_mean) / target_y_std
        if z_min > -spline_tail_bound or z_max < spline_tail_bound:
            raise RuntimeError(
                "Y_GRID endpoints do not both lie in the exact linear "
                "tails of the statistic spline."
            )
        exact_captured_flow_mass = float(
            norm.cdf(z_max) - norm.cdf(z_min)
        )
        exact_physical_flow_mass = float(norm.sf(z_min))
        exact_raw_upper_tail = float(norm.sf(z_max))
        numerical_flow_mass = np.exp(
            flow_physical_grid["log_normalization"]
        )
        raw_mass_error = float(np.max(np.abs(
            numerical_flow_mass - exact_captured_flow_mass
        )))

        coarse_y_grid = Y_GRID[::2]
        coarse_flow_mass = trapezoid(
            flow_physical_grid["density"][:, ::2],
            x=coarse_y_grid,
            axis=1,
        )
        coarse_hybrid_mass = trapezoid(
            hybrid1_grid["density"][:, ::2],
            x=coarse_y_grid,
            axis=1,
        )
        flow_half_integral_error = float(np.max(np.abs(
            coarse_flow_mass - 1.0
        )))
        hybrid_half_integral_error = float(np.max(np.abs(
            coarse_hybrid_mass - 1.0
        )))
        coarse_hybrid_cdf = cumulative_trapezoid(
            hybrid1_grid["density"][:, ::2],
            x=coarse_y_grid,
            axis=1,
            initial=0.0,
        ) / coarse_hybrid_mass[:, None]
        hybrid1_quantile_y = conditional_quantiles(
            hybrid1_grid["cdf"], Y_GRID, QUANTILE_LEVELS
        )
        coarse_hybrid1_quantile_y = conditional_quantiles(
            coarse_hybrid_cdf, coarse_y_grid, QUANTILE_LEVELS
        )
        y_resolution_shift = float(np.max(np.abs(
            coarse_hybrid1_quantile_y - hybrid1_quantile_y
        )))

        partial_hybrid_normalization = np.exp(
            hybrid1_grid["log_normalization"]
        )
        hybrid_upper_tail_bound = float(
            np.exp(LOG_RATIO_CLIP)
            * exact_raw_upper_tail
            / np.min(partial_hybrid_normalization)
        )
        print(
            f"Y attempt {support_attempt + 1}.{refinement + 1}: "
            f"Y_MAX={Y_MAX:.4f}, z_max={z_max:.3f}, "
            f"points={n_points:,}, max_dy={np.max(np.diff(Y_GRID)):.3e}; "
            f"q_shift={y_resolution_shift:.3e}, "
            f"half_mass(raw/hybrid)={flow_half_integral_error:.3e}/"
            f"{hybrid_half_integral_error:.3e}, "
            f"raw_exact_error={raw_mass_error:.3e}, "
            f"hybrid_tail_bound={hybrid_upper_tail_bound:.3e}"
        )

        if hybrid_upper_tail_bound > Y_UPPER_TAIL_MASS_TARGET:
            tail_requires_expansion = True
            break
        resolution_is_stable = (
            y_resolution_shift <= Y_QUANTILE_STABILITY_TARGET
            and flow_half_integral_error <= Y_HALF_INTEGRAL_TARGET
            and hybrid_half_integral_error <= Y_HALF_INTEGRAL_TARGET
            and raw_mass_error <= Y_RAW_MASS_TARGET
        )
        if resolution_is_stable:
            quadrature_converged = True
            break
        intervals *= 2

    if quadrature_converged:
        break
    if tail_requires_expansion:
        if support_attempt == Y_SUPPORT_MAX_EXPANSIONS:
            raise RuntimeError(
                "The analytic hybrid upper-tail bound did not converge. "
                "Inspect the first-ratio tail or increase the finite support."
            )
        Y_MAX += target_y_std
        continue
    raise RuntimeError(
        "Y quadrature did not pass the nested-grid stability checks after "
        f"{Y_RESOLUTION_MAX_REFINEMENTS} refinements."
    )

if not quadrature_converged:
    raise RuntimeError("Adaptive Y quadrature terminated without convergence.")
if (len(Y_GRID) - 1) % 2 or Y_GRID[::2][-1] != Y_MAX:
    raise RuntimeError("The final full/half Y grids are not exactly nested.")

hybrid1_quantile_t = np.maximum(
    np.exp(hybrid1_quantile_y) - T_OFFSET, 0.0
)
hybrid1_log_normalization_truncated = (
    hybrid1_grid["log_normalization"]
    - np.log(exact_physical_flow_mass)
)

# Rejection sampling removes only y < Y_MIN. Because Y_MIN lies in the exact
# linear tail, C_phi is analytic and cannot exceed one through quadrature bias.
flow_physical_mass = np.full(
    len(MU_DENSITY_GRID), exact_physical_flow_mass, dtype=np.float64
)
ratio1_acceptance = np.full(
    len(ratio1_mu), exact_physical_flow_mass, dtype=np.float64
)
quadrature_rejection = 1.0 - (
    len(ratio1_acceptance) / np.sum(1.0 / ratio1_acceptance)
)

hybrid1_edge_ratio = float(np.max(
    hybrid1_grid["density"][:, -1]
    / np.max(hybrid1_grid["density"], axis=1)
))
upper_buffer_index = int(np.searchsorted(
    Y_GRID, Y_MAX - Y_SUPPORT_BUFFER
))
hybrid1_upper_buffer_mass = float(np.max(
    1.0 - hybrid1_grid["cdf"][:, upper_buffer_index]
))
print(
    "Final Y-grid support/points/max spacing:",
    f"[{Y_MIN:.4f}, {Y_MAX:.4f}] / {len(Y_GRID):,} / "
    f"{np.max(np.diff(Y_GRID)):.3e}",
)
print(
    "Analytic physical flow mass / omitted raw/hybrid upper tail:",
    exact_physical_flow_mass,
    exact_raw_upper_tail,
    hybrid_upper_tail_bound,
)
print(
    "Final hybrid edge ratio / last-unit mass (diagnostic only):",
    hybrid1_edge_ratio,
    hybrid1_upper_buffer_mass,
)
print(
    "Flow rejection: observed / analytic context-matched =",
    f"{ratio1_rejection:.4%} / {quadrature_rejection:.4%}",
)
if abs(ratio1_rejection - quadrature_rejection) > 5.0e-3:
    raise RuntimeError(
        "Rejection sampling and the analytic physical-support mass disagree."
    )
print(
    "log Z1_tilde(mu) quantiles:",
    np.quantile(
        hybrid1_log_normalization_truncated, [0, .01, .5, .99, 1]
    ),
)
print(
    "r1 quadrature logit range / clipped fraction:",
    hybrid1_grid["ratio_log_range"][0],
    f"{hybrid1_grid['ratio_clip_fraction'][0]:.4%}",
)
print(
    "Full/half y-grid maximum quantile shift / fixed tolerance:",
    y_resolution_shift,
    Y_QUANTILE_STABILITY_TARGET,
)

if (
    np.max(np.abs(hybrid1_grid["cdf"][:, 0])) > 1.0e-10
    or np.max(np.abs(hybrid1_grid["cdf"][:, -1] - 1.0)) > 1.0e-10
    or np.min(np.diff(hybrid1_grid["cdf"], axis=1)) < -1.0e-10
):
    raise RuntimeError("F_H^(g) is not a valid conditional CDF.")


## 6. Freeze the construction simulator law; keep the audit law sealed

The construction template is the exact 250,000-event-per-process evaluation reservoir saved by Exercise 5. Before releasing the event networks, we reconstructed those deterministic-priority rows from the raw parquets and verified their compressed probabilities and bin counts against the saved arrays. The remaining selected evaluation rows were saved separately.

Those remaining rows are not merely a new pseudo-random toy seed. They are an **event-level complement** that was never used for PRESEL training, reference-flow training, ratio training, Exercise 5 density validation, the response map, or the PIT correction. Their compressed probabilities are not loaded until the final audit.

For scale only, we retain Exercise 5's random half-template discrepancy inside the construction reservoir. It is a warning about finite-template noise, not an alternative truth and not an audit.


In [ ]:
split_signal_l1 = np.sum(np.abs(
    saved_simulator_templates["calibration_signal_probability"]
    - saved_simulator_templates["audit_signal_probability"]
))
split_background_l1 = np.sum(np.abs(
    saved_simulator_templates["calibration_background_probability"]
    - saved_simulator_templates["audit_background_probability"]
))
print(
    "Construction simulator events:",
    f"background={SIM_CALIBRATION_EVENTS['background']:,}, "
    f"signal={SIM_CALIBRATION_EVENTS['signal']:,}",
)
print(
    "Diagnostic construction-half L1 differences: "
    f"signal={split_signal_l1:.4f}, "
    f"background={split_background_l1:.4f}"
)
print(
    "Exercise 5 ratio normalizers from",
    EXERCISE5_NORMALIZATION_SOURCE,
    EXERCISE5_RATIO_NORMALIZATION,
)
print(
    "Saved-to-frozen log-q scale correction:",
    f"{saved_simulator_templates['log_q_scale_correction']:+.6e}",
)
assert np.isclose(SIM_CALIBRATION_SIGNAL_PROBABILITY.sum(), 1.0)
assert np.isclose(SIM_CALIBRATION_BACKGROUND_PROBABILITY.sum(), 1.0)
print(
    "Audit cache validated without exposing its law:",
    SEALED_AUDIT_TEMPLATE_PATH.exists(),
)


## 7. Generate simulator-calibration pseudo-experiments

Simulator toys generate counts at physical $\mu$ under the construction template but evaluate the response-corrected numerator at $g(\mu)$. The main proposal is uniform on $[0,3]$. Because the previous auditor resolved the largest residual error at low $\mu$, we add a second matched proposal on $[0,0.75]$.

This oversampling does not change the conditional ratio target. Positive simulator PIT values and negative uniform PIT values carry exactly the same sampled $\mu_i$, so the nonuniform proposal density cancels from the classifier odds. It only allocates more calibration statistics where the earlier conditional estimate was weakest.


In [ ]:
simulator_calibration_uniform_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_construction_uniform_{N_SIMULATOR_CALIBRATION_TOYS}"
    ),
    n_toys=N_SIMULATOR_CALIBRATION_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 700,
    mu_range=MU_RANGE,
    signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
    background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)

simulator_calibration_low_mu_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_construction_low_mu_{N_SIMULATOR_LOW_MU_TOYS}"
    ),
    n_toys=N_SIMULATOR_LOW_MU_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 701,
    mu_range=LOW_MU_CALIBRATION_RANGE,
    signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
    background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)

simulator_calibration_toys = {
    name: np.concatenate([
        simulator_calibration_uniform_toys[name],
        simulator_calibration_low_mu_toys[name],
    ])
    for name in simulator_calibration_uniform_toys
}
N_SIMULATOR_CALIBRATION_TOTAL = len(simulator_calibration_toys["mu"])
expected_test_mu = pseudo_truth_response(
    simulator_calibration_toys["mu"]
)
if not np.allclose(
    simulator_calibration_toys["generation_mu"],
    simulator_calibration_toys["mu"],
    rtol=0.0,
    atol=2.0e-6,
) or not np.allclose(
    simulator_calibration_toys["test_mu"],
    expected_test_mu,
    rtol=0.0,
    atol=2.0e-6,
):
    raise RuntimeError(
        "The simulator calibration cache does not use physical generation "
        "and response-corrected testing."
    )
print(pd.DataFrame({
    "mu": simulator_calibration_toys["mu"],
    "g_mu": simulator_calibration_toys["test_mu"],
    "mu_hat": simulator_calibration_toys["mu_hat"],
    "mu_hat_minus_g": (
        simulator_calibration_toys["mu_hat"]
        - simulator_calibration_toys["test_mu"]
    ),
    "t_mu_g": simulator_calibration_toys["t_mu"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())

calibration_mu = simulator_calibration_toys["mu"].astype(np.float32)
calibration_y = np.log(
    simulator_calibration_toys["t_mu"].astype(np.float64) + T_OFFSET
)
below_y_grid = float(np.mean(calibration_y < Y_MIN))
above_y_grid = float(np.mean(calibration_y > Y_MAX))
print(
    "Simulator calibration toys outside Y_GRID (lower/upper):",
    f"{below_y_grid:.6%} / {above_y_grid:.6%}",
)
if below_y_grid > 0.0 or above_y_grid > 0.0:
    raise RuntimeError(
        "Simulator calibration statistics lie outside the certified Y_GRID. "
        "Increase its support and rerun before training the PIT ratio; "
        "clipping would create an artificial atom at u=0 or u=1."
    )


## 8. Learn the residual simulator ratio in the response-matched PIT coordinate

Map every simulator-calibration statistic through $F_{\rm H}^{(g)}$. Response matching should make this discrepancy appreciably smaller than in the identity-response run, but no equality is assumed. A matched classifier compares the simulator $u_0$ with a uniform draw at the same physical $\mu$ and estimates the entire residual conditional density, not only its 95% quantile.


In [ ]:
calibration_u0_raw = conditional_cdf_values(
    hybrid1_grid["cdf"],
    MU_DENSITY_GRID,
    Y_GRID,
    calibration_mu,
    calibration_y,
)
exact_boundary_fraction = float(np.mean(
    (calibration_u0_raw <= 0.0) | (calibration_u0_raw >= 1.0)
))
print(f"Exact PIT-boundary fraction: {exact_boundary_fraction:.6%}")
if exact_boundary_fraction > 0.0:
    raise RuntimeError(
        "Interior simulator PIT values reached exactly 0 or 1. "
        "Inspect CDF support/interpolation before training."
    )
calibration_u0 = np.clip(
    calibration_u0_raw, PIT_EPS, 1.0 - PIT_EPS
).astype(np.float32)
pit_clipped_fraction = float(np.mean(
    (calibration_u0_raw < PIT_EPS)
    | (calibration_u0_raw > 1.0 - PIT_EPS)
))
print(
    f"PIT values moved by the numerical {PIT_EPS:g} clip: "
    f"{pit_clipped_fraction:.6%}"
)

pit_rng = np.random.default_rng(SEED + 800)
calibration_u_reference = np.clip(
    pit_rng.uniform(0.0, 1.0, size=N_SIMULATOR_CALIBRATION_TOTAL),
    PIT_EPS,
    1.0 - PIT_EPS,
).astype(np.float32)
calibration_positive = np.column_stack([
    calibration_mu, calibration_u0
])
calibration_negative = np.column_stack([
    calibration_mu, calibration_u_reference
])
if not np.array_equal(
    calibration_positive[:, 0], calibration_negative[:, 0]
):
    raise RuntimeError("The PIT classifier lost its matched mu design.")

paired_calibration_ids = np.arange(
    N_SIMULATOR_CALIBRATION_TOTAL, dtype=np.int64
)
calibration_ratio_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(
        f"Training PIT calibration member "
        f"{member + 1}/{RATIO_ENSEMBLE_SIZE}"
    )
    print("=" * 76)
    calibration_ratio_ensemble.append(
        train_ratio_classifier(
            calibration_positive,
            calibration_negative,
            checkpoint=(
                CALIBRATION_RATIO_MODEL_DIR
                / f"r_cal_member{member}.pt"
            ),
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 810 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_calibration_ids,
            verify_checkpoint_data=True,
        )
    )
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in calibration_ratio_ensemble
)
assert not any(
    pack["history"].get("weighted_bce", False)
    for pack in calibration_ratio_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(calibration_ratio_ensemble):
    ax.plot(
        pack.get("history", {}).get("validation", []),
        label=f"member {member}",
    )
ax.axhline(np.log(2.0), color="black", ls="--", lw=1.2,
           label=r"$\log 2$")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="Residual simulator correction after response matching")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "pit_ratio_training")
plt.show()


## 9. Normalize the residual PIT ratio and build the calibrated primitive

The PIT reference is exactly uniform, so the only remaining normalizer is the one-dimensional integral over $u\in[0,1]$:

$$
G(u\mid\mu)=
\frac{\int_0^u r_{\rm cal}(v,\mu)\,dv}
     {\int_0^1 r_{\rm cal}(v,\mu)\,dv}.
$$

The code checks logit clipping, normalization, monotonicity, and nested full/half-grid stability before the map can be used.


In [ ]:
U_GRID = np.linspace(0.0, 1.0, PIT_QUADRATURE_POINTS)
calibration_grid = conditional_ratio_grid(
    calibration_ratio_ensemble,
    MU_DENSITY_GRID,
    U_GRID,
    max_abs_log_ratio=LOG_RATIO_CLIP,
)
print(
    "raw log Z_cal(mu) quantiles:",
    np.quantile(
        calibration_grid["log_normalization"],
        [0, .01, .5, .99, 1],
    ),
)
print(
    "PIT-ratio logit range / clipped fraction:",
    calibration_grid["ratio_log_range"],
    f"{calibration_grid['ratio_clip_fraction']:.4%}",
)
if (
    np.max(np.abs(calibration_grid["cdf"][:, 0])) > 1.0e-10
    or np.max(np.abs(calibration_grid["cdf"][:, -1] - 1.0))
    > 1.0e-10
    or np.min(np.diff(calibration_grid["cdf"], axis=1)) < -1.0e-10
):
    raise RuntimeError("G is not a valid conditional CDF.")

def evaluate_calibrated_pit(mu, t_mu):
    """Return (U0, Ucal) for arbitrary interior toy/statistic rows."""
    mu = np.asarray(mu, dtype=np.float64).reshape(-1)
    t_mu = np.asarray(t_mu, dtype=np.float64).reshape(-1)
    if len(mu) != len(t_mu) or np.any(t_mu < 0.0):
        raise ValueError("mu/t_mu must be matched and t_mu non-negative.")
    if np.any(mu <= MU_RANGE[0]) or np.any(mu >= MU_RANGE[1]):
        raise ValueError("Use the empirical rules at exact endpoints.")
    y = np.log(t_mu + T_OFFSET)
    if np.any(y < Y_MIN) or np.any(y > Y_MAX):
        raise ValueError("Statistic values lie outside the frozen Y_GRID.")
    u0 = conditional_cdf_values(
        hybrid1_grid["cdf"], MU_DENSITY_GRID, Y_GRID, mu, y
    )
    u_cal = conditional_cdf_values(
        calibration_grid["cdf"], MU_DENSITY_GRID, U_GRID, mu, u0
    )
    return u0, u_cal


def evaluate_calibrated_statistic(mu, t_mu):
    """Map Ucal to a chi-square-looking scalar statistic."""
    _, u_cal = evaluate_calibrated_pit(mu, t_mu)
    return chi2.ppf(np.clip(u_cal, 1.0e-12, 1.0 - 1.0e-12), df=1)


_, calibration_u_cal = evaluate_calibrated_pit(
    calibration_mu, simulator_calibration_toys["t_mu"]
)

selected_mu = [0.25, 1.0, 2.0, 2.75]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for mu_value in selected_mu:
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    axes[0].plot(
        U_GRID,
        calibration_grid["density"][index],
        lw=1.8,
        label=rf"$\mu={mu_value:g}$",
    )
    axes[1].plot(
        U_GRID,
        calibration_grid["cdf"][index],
        lw=1.8,
        label=rf"$\mu={mu_value:g}$",
    )
axes[0].axhline(1.0, color="black", ls="--", lw=1.2,
                label="Uniform density")
axes[1].plot(U_GRID, U_GRID, "k--", lw=1.2,
             label="Identity / no correction")
axes[0].set(xlabel=r"response-matched PIT $u_0$", ylabel=r"$g_{\rm cal}(u_0\mid\mu)$",
            title="Learned simulator/hNDE ratio")
axes[1].set(xlabel=r"response-matched PIT $u_0$", ylabel=r"$G(u_0\mid\mu)$",
            title="Conditional calibration map")
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
fig.tight_layout()
export_exercise11_figure(fig, "pit_ratio_and_primitive")
plt.show()

bins = np.linspace(0.0, 1.0, 41)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
axes[0].hist(calibration_u0_raw, bins=bins, density=True,
             histtype="step", lw=2, color="C1")
axes[1].hist(calibration_u_cal, bins=bins, density=True,
             histtype="step", lw=2, color="C0")
for ax in axes:
    ax.axhline(1.0, color="black", ls="--", lw=1.2)
    ax.set(xlim=(0, 1), xlabel="PIT value", ylabel="Density")
    ax.grid(alpha=.2)
axes[0].set_title(r"Before calibration: $U_0=F_{\rm H}(T\mid\mu)$")
axes[1].set_title(r"After calibration: $U_{\rm cal}=G(U_0\mid\mu)$")
fig.tight_layout()
export_exercise11_figure(fig, "pit_calibration_training_closure")
plt.show()


## 10. Conditional quantiles, endpoints, and Neyman inversion

The calibrated $\gamma$ quantile follows without sampling the learned density:

$$
u_\gamma(\mu)=G^{-1}(\gamma\mid\mu),
\qquad
c_\gamma(\mu)
=\left(F_{\rm H}^{(g)}\right)^{-1}
\!\left(u_\gamma(\mu)\mid\mu\right).
$$

We compare these curves with the response-matched hNDE quantiles and with direct, coarse binned simulator quantiles. The latter use the calibration ensemble and are therefore only a construction diagnostic.

### Exact endpoints

The continuous proposal draws no point exactly at $\mu=0$ or $3$, while the compressed Poisson statistic remains formally discrete. Under the old identity response, $g(0)=0$ and the constrained MLE created a large atom at $t_0=0$. After response matching that specific atom persists only if $g(0)=0$; it must not be assumed. Nevertheless, neither continuous interpolation nor a continuous density-ratio model is the right tool for an exact endpoint that received no training mass.

We therefore generate explicit construction-template endpoint ensembles and use the finite-sample conservative order statistic

$$
k=\left\lceil\gamma(n+1)\right\rceil,
\qquad c_\gamma=T_{(k)}.
$$

Ties can make this non-randomized endpoint test conservative. Endpoint quantiles are treated piecewise and are not inserted into the smooth interior cutoff interpolant. The helper below applies calibrated PIT inversion on the open interval and empirical cutoffs at the two exact endpoints.


In [ ]:
calibration_u0_quantiles = conditional_quantiles(
    calibration_grid["cdf"], U_GRID, QUANTILE_LEVELS
)
calibrated_quantile_y = conditional_row_quantiles(
    hybrid1_grid["cdf"],
    Y_GRID,
    calibration_u0_quantiles,
)
calibrated_quantile_t = np.maximum(
    np.exp(calibrated_quantile_y) - T_OFFSET, 0.0
)

coarse_u_grid = U_GRID[::2]
coarse_calibration_density = calibration_grid["density"][:, ::2]
coarse_calibration_normalization = trapezoid(
    coarse_calibration_density, x=coarse_u_grid, axis=1
)
coarse_calibration_cdf = cumulative_trapezoid(
    coarse_calibration_density,
    x=coarse_u_grid,
    axis=1,
    initial=0.0,
) / coarse_calibration_normalization[:, None]
coarse_calibration_u0_quantiles = conditional_quantiles(
    coarse_calibration_cdf, coarse_u_grid, QUANTILE_LEVELS
)
coarse_calibrated_quantile_y = conditional_row_quantiles(
    hybrid1_grid["cdf"],
    Y_GRID,
    coarse_calibration_u0_quantiles,
)
u_resolution_shift = float(np.max(np.abs(
    coarse_calibrated_quantile_y - calibrated_quantile_y
)))
u_resolution_tolerance = max(0.02, 4.0 * np.max(np.diff(Y_GRID)))
print(
    "Full/half u-grid maximum final-quantile shift / tolerance:",
    u_resolution_shift, u_resolution_tolerance,
)
if u_resolution_shift > u_resolution_tolerance:
    raise RuntimeError(
        "The calibrated conditional quantiles are not stable when "
        "the u-grid resolution is halved. Increase "
        "PIT_QUADRATURE_POINTS."
    )

calibrated_cdf_grid = np.asarray([
    np.interp(f_h_row, U_GRID, g_row)
    for f_h_row, g_row in zip(
        hybrid1_grid["cdf"], calibration_grid["cdf"]
    )
])
composed_quantile_y = conditional_quantiles(
    calibrated_cdf_grid, Y_GRID, QUANTILE_LEVELS
)
composition_difference = np.max(np.abs(
    composed_quantile_y - calibrated_quantile_y
))
print(
    "Maximum nested/composed quantile difference in y:",
    composition_difference,
)
if composition_difference > 2.0 * np.max(np.diff(Y_GRID)):
    raise RuntimeError("Nested and composed conditional CDFs disagree.")

index_95 = int(np.flatnonzero(
    np.isclose(QUANTILE_LEVELS, .95)
)[0])
critical_hnde_grid = hybrid1_quantile_t[:, index_95]
critical_calibrated_grid = calibrated_quantile_t[:, index_95]
critical_calibrated_smooth = PchipInterpolator(
    MU_DENSITY_GRID, critical_calibrated_grid, extrapolate=False
)

anchor_hnde = {}
anchor_simulator = {}
anchor_hnde_quantiles = []
anchor_simulator_quantiles = []
for anchor_mu in ANCHOR_MUS:
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_hnde[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=CACHE_DIR / f"anchor_hnde_{key}_{N_ANCHOR_TOYS}",
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 900 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        generation_mu_transform=pseudo_truth_response,
        test_mu_transform=pseudo_truth_response,
        generation_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    anchor_simulator[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"anchor_simulator_construction_{key}_{N_ANCHOR_TOYS}"
        ),
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 910 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
        background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        test_mu_transform=pseudo_truth_response,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    anchor_hnde_quantiles.append(conservative_empirical_quantile(
        anchor_hnde[anchor_mu]["t_mu"], QUANTILE_LEVELS
    ))
    anchor_simulator_quantiles.append(conservative_empirical_quantile(
        anchor_simulator[anchor_mu]["t_mu"], QUANTILE_LEVELS
    ))
anchor_hnde_quantiles = np.asarray(anchor_hnde_quantiles)
anchor_simulator_quantiles = np.asarray(anchor_simulator_quantiles)

def neyman_accepts_95(mu, t_mu):
    """Apply the endpoint-aware rule to response-corrected t_mu^(g)."""
    mu, t_mu = np.broadcast_arrays(
        np.asarray(mu, dtype=np.float64),
        np.asarray(t_mu, dtype=np.float64),
    )
    output_shape = mu.shape
    mu = mu.reshape(-1)
    t_mu = t_mu.reshape(-1)
    if np.any(t_mu < 0.0) or not np.isfinite(t_mu).all():
        raise ValueError("t_mu must be finite and non-negative.")
    if (
        not np.isfinite(mu).all()
        or np.any(mu < MU_RANGE[0])
        or np.any(mu > MU_RANGE[1])
    ):
        raise ValueError("mu lies outside the calibrated parameter range.")
    accepted = np.empty(len(mu), dtype=bool)
    assigned = np.zeros(len(mu), dtype=bool)
    interior = (mu > MU_RANGE[0]) & (mu < MU_RANGE[1])
    if np.any(interior):
        _, u_cal = evaluate_calibrated_pit(mu[interior], t_mu[interior])
        accepted[interior] = u_cal <= .95
        assigned[interior] = True
    for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
        endpoint = mu == anchor_mu
        accepted[endpoint] = (
            t_mu[endpoint]
            <= anchor_simulator_quantiles[anchor_index, index_95]
        )
        assigned[endpoint] = True
    if not np.all(assigned):
        raise ValueError("mu lies outside the calibrated parameter range.")
    return accepted.reshape(output_shape)

calibration_quantile_edges = np.linspace(*MU_RANGE, 21)
calibration_quantile_centers = 0.5 * (
    calibration_quantile_edges[:-1] + calibration_quantile_edges[1:]
)
calibration_bin_index = np.clip(
    np.digitize(calibration_mu, calibration_quantile_edges) - 1,
    0,
    len(calibration_quantile_centers) - 1,
)
direct_calibration_q95 = np.asarray([
    conservative_empirical_quantile(
        simulator_calibration_toys["t_mu"][calibration_bin_index == i],
        .95,
    )
    for i in range(len(calibration_quantile_centers))
])

np.savez_compressed(
    PIT_CACHE_DIR / "conditional_pit_calibration.npz",
    mu=MU_DENSITY_GRID,
    levels=QUANTILE_LEVELS,
    hnde=hybrid1_quantile_t,
    simulator_calibrated=calibrated_quantile_t,
    endpoint_mu=ANCHOR_MUS,
    endpoint_hnde=anchor_hnde_quantiles,
    endpoint_simulator=anchor_simulator_quantiles,
    u_grid=U_GRID,
    calibration_cdf=calibration_grid["cdf"],
    response_mu=RESPONSE_MU_GRID,
    response_g=response_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
colors = plt.cm.viridis(np.linspace(.12, .9, len(QUANTILE_LEVELS)))
interior_grid = slice(1, -1)
for level, color, before, after in zip(
    QUANTILE_LEVELS,
    colors,
    hybrid1_quantile_t.T,
    calibrated_quantile_t.T,
):
    linewidth = 2.8 if np.isclose(level, .95) else 1.25
    axes[0].plot(
        MU_DENSITY_GRID[interior_grid], before[interior_grid],
        color=color, ls="--", lw=linewidth,
        label=(rf"{level:.0%} response-matched hNDE"
               if np.isclose(level, .95)
               else rf"{level:.0%}"),
    )
    axes[0].plot(
        MU_DENSITY_GRID[interior_grid], after[interior_grid],
        color=color, lw=linewidth,
        label=(rf"{level:.0%} calibrated"
               if np.isclose(level, .95) else None),
    )
    axes[1].plot(
        MU_DENSITY_GRID[interior_grid],
        (after - before)[interior_grid],
        color=color, lw=linewidth, label=rf"{level:.0%}",
    )
axes[0].scatter(
    ANCHOR_MUS,
    anchor_simulator_quantiles[:, index_95],
    marker="s", s=42, color="C3", zorder=5,
    label="empirical endpoint 95%",
)
axes[0].scatter(
    calibration_quantile_centers,
    direct_calibration_q95,
    marker="o", s=18, facecolors="none", edgecolors="0.25",
    label="binned simulator 95%",
)
axes[0].set(
    xlabel=r"$\mu$", ylabel=r"conditional quantile of $t_\mu$",
    title="Response-matched baseline and residual PIT calibration",
)
axes[1].axhline(0.0, color="black", ls=":", lw=1)
axes[1].set(
    xlabel=r"$\mu$", ylabel="calibrated − hNDE quantile",
    title="Calibration displacement",
)
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "conditional_pit_quantile_corrections")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0),
                         sharex=True, sharey=True)
t_grid = np.maximum(np.exp(Y_GRID) - T_OFFSET, 0.0)
for ax, mu_value in zip(axes.flat, [0.25, 1.0, 2.0, 2.75]):
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    ax.plot(t_grid, hybrid1_grid["cdf"][index], ls="--", lw=2,
            label="response-matched hNDE")
    ax.plot(t_grid, calibrated_cdf_grid[index], lw=2,
            label="PIT-ratio calibrated")
    ax.axhline(.95, color="0.4", ls=":", lw=1)
    ax.set_xlim(
        0, max(8.0, float(critical_calibrated_grid[index]) * 1.5)
    )
    ax.set_title(rf"$\mu={mu_value:g}$")
    ax.grid(alpha=.2)
for ax in axes[-1]:
    ax.set_xlabel(r"$t_\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel("Conditional CDF")
axes[0, 0].legend()
fig.tight_layout()
export_exercise11_figure(fig, "conditional_pit_cdf_primitives")
plt.show()


## 11. Same-template internal audit: isolate calibration error

Before opening the unused event reservoir, generate a fresh toy ensemble from the same fixed construction template used by $g$ and by the PIT ratio. These toys have new random seeds and were not used for training. This control answers a narrow question: **does the learned conditional calibration reproduce its own declared construction law?**

It does not test event-level simulator transfer or finite-template uncertainty. Keeping this control separate is useful: a failure here is calibration-model error, whereas a discrepancy that appears only after the sealed audit is opened is evidence of sensitivity to the finite construction reservoir.


In [ ]:
simulator_internal_audit_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_construction_internal_audit_"
          f"{N_SIMULATOR_INTERNAL_AUDIT_TOYS}"
    ),
    n_toys=N_SIMULATOR_INTERNAL_AUDIT_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 1000,
    mu_range=MU_RANGE,
    signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
    background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)
internal_mu = simulator_internal_audit_toys["mu"].astype(np.float64)
internal_t = simulator_internal_audit_toys["t_mu"].astype(np.float64)
if not np.allclose(
    simulator_internal_audit_toys["generation_mu"], internal_mu,
    rtol=0.0, atol=2.0e-6,
) or not np.allclose(
    simulator_internal_audit_toys["test_mu"],
    pseudo_truth_response(internal_mu),
    rtol=0.0, atol=2.0e-6,
):
    raise RuntimeError("The internal-audit cache has stale coordinates.")
internal_u0, internal_u_cal = evaluate_calibrated_pit(
    internal_mu, internal_t
)
internal_covered_hnde = internal_u0 <= .95
internal_covered_calibrated = internal_u_cal <= .95
if not np.array_equal(
    neyman_accepts_95(internal_mu, internal_t),
    internal_covered_calibrated,
):
    raise RuntimeError(
        "Endpoint-aware decisions disagree in the open-interval internal audit."
    )
coverage_edges = np.linspace(*MU_RANGE, 21)
binned_internal = binned_coverage(
    internal_mu, internal_covered_calibrated, edges=coverage_edges
)
print(
    "Internal same-template response-baseline coverage:",
    f"{internal_covered_hnde.mean():.4%}",
)
print(
    "Internal same-template PIT-calibrated coverage:",
    f"{internal_covered_calibrated.mean():.4%}",
)
print(
    "Internal calibrated binned range:",
    f"[{binned_internal['coverage'].min():.4%}, "
    f"{binned_internal['coverage'].max():.4%}]",
)


## 12. Finite-construction-template bootstrap diagnostic

The calibration reservoir contains a finite number of simulator events. To measure how much the *fixed* construction changes when that empirical law fluctuates, resample the signal and background compressed bin counts multinomially, generate toys under each bootstrap template, and apply the already frozen response map and calibrated pivot.

This is a sensitivity diagnostic, not another independent coverage audit. Every replicate is derived from the same construction events. Its spread combines template sensitivity with finite toy noise; Wilson intervals show the latter for each replicate. Recalibrating separately inside every bootstrap would answer a different hierarchical question and would not test the robustness of this fixed construction.


In [ ]:
def multinomial_template_bootstrap(counts, rng):
    counts = np.asarray(counts, dtype=np.int64)
    if np.any(counts < 0) or counts.sum() <= 0:
        raise ValueError("Bootstrap template counts must be non-negative.")
    draw = rng.multinomial(
        int(counts.sum()), counts / counts.sum()
    ).astype(np.float64)
    draw += 1.0e-15
    return draw / draw.sum()


template_bootstrap_rows = []
template_bootstrap_binned = []
bootstrap_edges = np.linspace(*MU_RANGE, 11)
for replicate in range(N_TEMPLATE_BOOTSTRAPS):
    rng = np.random.default_rng(
        np.random.SeedSequence([SEED, 1200, replicate])
    )
    bootstrap_signal_probability = multinomial_template_bootstrap(
        SIM_CALIBRATION_SIGNAL_COUNTS, rng
    )
    bootstrap_background_probability = multinomial_template_bootstrap(
        SIM_CALIBRATION_BACKGROUND_COUNTS, rng
    )
    bootstrap_toys = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"construction_template_bootstrap_{replicate:03d}_"
              f"{N_TOYS_PER_TEMPLATE_BOOTSTRAP}"
        ),
        n_toys=N_TOYS_PER_TEMPLATE_BOOTSTRAP,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 1300 + replicate,
        mu_range=MU_RANGE,
        signal_probability=bootstrap_signal_probability,
        background_probability=bootstrap_background_probability,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        test_mu_transform=pseudo_truth_response,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    bootstrap_covered = neyman_accepts_95(
        bootstrap_toys["mu"], bootstrap_toys["t_mu"]
    )
    successes = int(bootstrap_covered.sum())
    lower, upper = wilson_interval(successes, len(bootstrap_covered))
    template_bootstrap_rows.append({
        "replicate": replicate,
        "coverage": float(bootstrap_covered.mean()),
        "toy_wilson_lower": float(lower),
        "toy_wilson_upper": float(upper),
    })
    template_bootstrap_binned.append(
        binned_coverage(
            bootstrap_toys["mu"],
            bootstrap_covered,
            edges=bootstrap_edges,
        )["coverage"]
    )

template_bootstrap_table = pd.DataFrame(template_bootstrap_rows)
template_bootstrap_binned = np.asarray(template_bootstrap_binned)
bootstrap_global_interval = np.quantile(
    template_bootstrap_table["coverage"], [.025, .5, .975]
)
bootstrap_binned_interval = np.quantile(
    template_bootstrap_binned, [.025, .5, .975], axis=0
)
display(
    template_bootstrap_table.describe().T.style.format(precision=6)
)
print(
    "Construction-template bootstrap global 2.5/50/97.5% quantiles:",
    bootstrap_global_interval,
)
print(
    "Reminder: the construction half-template L1 scale was "
    f"signal={split_signal_l1:.4f}, background={split_background_l1:.4f}."
)


## 13. Open the sealed event-level reservoir and perform the final LF2I audit

Only now—after $g$, $F_{\rm H}^{(g)}$, the PIT ratio, numerical grids, endpoint cutoffs, the internal control, and the bootstrap diagnostic are frozen—do we expose the audit probability law. Earlier cache checks validated only its declared recipe, shape, normalization, counts, and shared pair fingerprint; no audit value entered the construction.

Its signal and background events are the deterministic-priority complement of Exercise 5's retained evaluation reservoir. The final toys are therefore independent at two levels: they use new Poisson pseudo-experiment seeds, and their empirical simulator law comes from event rows never used by the construction. Each toy is still generated at physical $\mu$ and evaluated with $t_\mu^{(g)}$.

The BCE auditor receives only $\mu$ and estimates

$$
a^*(\mu)=\Pr_{\rm fresh\ event\ reservoir}
\!\left[U_{\rm cal}\leq0.95\mid\mu\right].
$$

Equal-width bin estimates with Wilson intervals prevent a smooth auditor from hiding localized departures. The direct pivot decision is canonical; the interpolated raw-statistic cutoff is checked only as a numerical equivalence diagnostic.

This audit is consumable. If its result is used to alter the construction again, these events are no longer fresh for the revised coverage claim; a new simulator sample is then required.


In [ ]:
with np.load(SEALED_AUDIT_TEMPLATE_PATH) as audit_template:
    if not np.array_equal(audit_template["log_q_edges"], LOG_Q_EDGES):
        raise RuntimeError("The sealed audit template has stale q binning.")
    if str(
        audit_template["reservoir_pair_fingerprint"].item()
    ) != RESERVOIR_PAIR_FINGERPRINT:
        raise RuntimeError(
            "The construction/audit reservoir pair fingerprint changed."
        )
    if not np.isclose(
        float(audit_template["presel_ratio_cut"]), PRESEL_RATIO_CUT,
        rtol=0.0, atol=1.0e-12,
    ):
        raise RuntimeError("The sealed audit template has stale PRESEL state.")
    for process in ("signal", "background"):
        if not np.isclose(
            float(audit_template[f"{process}_ratio_normalization"]),
            RATIO_NORMALIZATION[process],
            rtol=0.0, atol=1.0e-12,
        ):
            raise RuntimeError(
                f"The sealed audit template has stale {process} "
                "ratio normalization."
            )
    SIM_AUDIT_SIGNAL_PROBABILITY = audit_template[
        "signal_probability"
    ].astype(np.float64)
    SIM_AUDIT_BACKGROUND_PROBABILITY = audit_template[
        "background_probability"
    ].astype(np.float64)
    SIM_AUDIT_EVENTS = {
        "signal": int(audit_template["signal_events"]),
        "background": int(audit_template["background_events"]),
    }
expected_audit_events = {
    process: EXERCISE5_EXPECTED_SELECTED_EVAL[process]
    - EXERCISE5_EVAL_CAP
    for process in ("signal", "background")
}
if SIM_AUDIT_EVENTS != expected_audit_events:
    raise RuntimeError(
        "The sealed audit event counts do not match the certified "
        "Exercise 5 complement."
    )
for probability in (
    SIM_AUDIT_SIGNAL_PROBABILITY,
    SIM_AUDIT_BACKGROUND_PROBABILITY,
):
    if not np.isfinite(probability).all() or not np.isclose(
        probability.sum(), 1.0
    ):
        raise RuntimeError("A sealed audit probability is invalid.")

audit_signal_l1 = float(np.sum(np.abs(
    SIM_AUDIT_SIGNAL_PROBABILITY
    - SIM_CALIBRATION_SIGNAL_PROBABILITY
)))
audit_background_l1 = float(np.sum(np.abs(
    SIM_AUDIT_BACKGROUND_PROBABILITY
    - SIM_CALIBRATION_BACKGROUND_PROBABILITY
)))
print("Opened sealed unused-event reservoir:", SIM_AUDIT_EVENTS)
print(
    "Construction/audit event-template L1 differences:",
    f"signal={audit_signal_l1:.4f}, background={audit_background_l1:.4f}",
)

simulator_audit_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_unused_event_audit_{N_SIMULATOR_AUDIT_TOYS}"
    ),
    n_toys=N_SIMULATOR_AUDIT_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 1400,
    mu_range=MU_RANGE,
    signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
    background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)
audit_mu = simulator_audit_toys["mu"].astype(np.float64)
audit_t = simulator_audit_toys["t_mu"].astype(np.float64)
if not np.allclose(
    simulator_audit_toys["generation_mu"], audit_mu,
    rtol=0.0, atol=2.0e-6,
) or not np.allclose(
    simulator_audit_toys["test_mu"], pseudo_truth_response(audit_mu),
    rtol=0.0, atol=2.0e-6,
):
    raise RuntimeError("The sealed-audit toy cache has stale coordinates.")

audit_u0, audit_u_cal = evaluate_calibrated_pit(audit_mu, audit_t)
covered_hnde = audit_u0 <= .95
covered_calibrated = audit_u_cal <= .95
if not np.array_equal(
    neyman_accepts_95(audit_mu, audit_t), covered_calibrated
):
    raise RuntimeError(
        "The endpoint-aware helper disagrees with the interior PIT rule."
    )

audit_critical_calibrated = critical_calibrated_smooth(audit_mu)
covered_by_cutoff = audit_t <= audit_critical_calibrated
cutoff_disagreement = float(np.mean(
    covered_by_cutoff != covered_calibrated
))
print(f"Pivot/cutoff decision disagreement: {cutoff_disagreement:.4%}")
if cutoff_disagreement > 2.0e-3:
    print(
        "WARNING: the auxiliary interpolated cutoff disagrees with more "
        "than 0.2% of direct pivot decisions. Refine the mu/y/u grids "
        "before using the smooth cutoff as a numerical substitute."
    )

audit_t_cal = evaluate_calibrated_statistic(audit_mu, audit_t)
print(
    "Fresh-reservoir response-baseline coverage:",
    f"{covered_hnde.mean():.4%}",
)
print(
    "Fresh-reservoir PIT-calibrated coverage:",
    f"{covered_calibrated.mean():.4%}",
)
print(
    "Equivalent chi-square decision coverage:",
    f"{np.mean(audit_t_cal <= chi2.ppf(.95, df=1)):.4%}",
)

coverage_auditor = train_coverage_auditor(
    audit_mu,
    covered_calibrated,
    checkpoint=PIT_MODEL_DIR / "coverage_auditor_unused_events_95cl.pt",
    model_config=AUDITOR_MODEL_CONFIG,
    training_config=AUDITOR_TRAINING_CONFIG,
    device=device,
    seed=SEED + 1410,
    load_if_available=LOAD_IF_AVAILABLE,
)
auditor_grid = coverage_auditor_probability(
    coverage_auditor, MU_DENSITY_GRID
)
null_bce = -0.95 * np.log(.95) - 0.05 * np.log(.05)
print(f"Bernoulli(0.95) null BCE: {null_bce:.6f}")
print(
    "Fresh-reservoir auditor predicted-coverage range:",
    f"[{auditor_grid.min():.4%}, {auditor_grid.max():.4%}]",
)


In [ ]:
binned_before = binned_coverage(
    audit_mu, covered_hnde, edges=coverage_edges
)

binned_after = binned_coverage(
    audit_mu, covered_calibrated, edges=coverage_edges
)

anchor_audit_rows = []
for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_audit = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"anchor_simulator_unused_event_audit_"
              f"{key}_{N_AUDIT_ANCHOR_TOYS}"
        ),
        n_toys=N_AUDIT_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 1500 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
        background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        test_mu_transform=pseudo_truth_response,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    endpoint_critical = float(
        anchor_simulator_quantiles[anchor_index, index_95]
    )
    endpoint_covered = neyman_accepts_95(
        np.full(N_AUDIT_ANCHOR_TOYS, anchor_mu),
        anchor_audit["t_mu"],
    )
    successes = int(endpoint_covered.sum())
    lower, upper = wilson_interval(successes, len(endpoint_covered))
    anchor_audit_rows.append({
        "mu": anchor_mu,
        "construction_critical_value": endpoint_critical,
        "fresh_reservoir_coverage": float(endpoint_covered.mean()),
        "wilson_lower": float(lower),
        "wilson_upper": float(upper),
        "zero_mass": float(np.mean(
            anchor_audit["t_mu"] <= 1.0e-12
        )),
    })
anchor_audit_table = pd.DataFrame(anchor_audit_rows)
display(anchor_audit_table.style.format(precision=5).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8.8, 5.6))
ax.axhline(.95, color="black", ls="--", lw=1.5,
           label="Nominal 95%")
ax.plot(MU_DENSITY_GRID, auditor_grid, color="C3", lw=2.3,
        label="fresh-reservoir BCE auditor")
ax.errorbar(
    binned_before["center"], binned_before["coverage"],
    yerr=[
        binned_before["coverage"] - binned_before["lower"],
        binned_before["upper"] - binned_before["coverage"],
    ],
    fmt="o", ms=4, color="0.5", alpha=.75,
    label="response baseline, fresh reservoir",
)
ax.errorbar(
    binned_internal["center"], binned_internal["coverage"],
    yerr=[
        binned_internal["coverage"] - binned_internal["lower"],
        binned_internal["upper"] - binned_internal["coverage"],
    ],
    fmt="D", ms=4, color="C1", alpha=.8,
    label="calibrated, same-template control",
)
ax.errorbar(
    binned_after["center"], binned_after["coverage"],
    yerr=[
        binned_after["coverage"] - binned_after["lower"],
        binned_after["upper"] - binned_after["coverage"],
    ],
    fmt="o", ms=5, color="C0",
    label="calibrated, fresh event reservoir",
)
ax.errorbar(
    anchor_audit_table["mu"],
    anchor_audit_table["fresh_reservoir_coverage"],
    yerr=[
        anchor_audit_table["fresh_reservoir_coverage"]
        - anchor_audit_table["wilson_lower"],
        anchor_audit_table["wilson_upper"]
        - anchor_audit_table["fresh_reservoir_coverage"],
    ],
    fmt="s", ms=6, color="C2",
    label="fresh-reservoir endpoint anchors",
)
ax.set(
    xlim=MU_RANGE,
    xlabel=r"physical truth $\mu$",
    ylabel="Conditional coverage",
    title="Sealed event-level LF2I coverage audit",
)
ax.set_ylim(min(.90, binned_before["lower"].min() - .005), 1.005)
ax.grid(alpha=.2)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "lf2i_response_pit_sealed_event_audit")
plt.show()

coverage_summary = pd.DataFrame({
    "method": [
        "response baseline — fresh reservoir",
        "PIT calibrated — same-template control",
        "PIT calibrated — fresh event reservoir",
    ],
    "global_coverage": [
        covered_hnde.mean(),
        internal_covered_calibrated.mean(),
        covered_calibrated.mean(),
    ],
    "minimum_binned_coverage": [
        binned_before["coverage"].min(),
        binned_internal["coverage"].min(),
        binned_after["coverage"].min(),
    ],
    "maximum_binned_coverage": [
        binned_before["coverage"].max(),
        binned_internal["coverage"].max(),
        binned_after["coverage"].max(),
    ],
})
display(coverage_summary.style.format(precision=5).hide(axis="index"))
print(
    "Construction-template bootstrap global 95% sensitivity interval:",
    bootstrap_global_interval[[0, 2]],
)


## 14. Invert the pivot for pseudo-observed datasets

Coverage is only one property of a Neyman construction; calibration can change the geometry and length of the confidence set. We now generate several illustrative pseudo-observed datasets from the already opened unused-event reservoir, evaluate $t_\mu^{(g)}$ over a dense physical-$\mu$ scan, and invert two rules:

1. the response-matched hNDE baseline, $F_{\rm H}^{(g)}(t_\mu^{(g)}\mid\mu)\leq0.95$;
2. the residual simulator-calibrated rule, $G(F_{\rm H}^{(g)}(t_\mu^{(g)}\mid\mu)\mid\mu)\leq0.95$.

Both inversions use their own empirical endpoint cutoffs. We report total confidence-set length within the calibrated design interval $[0,3]$ and preserve all connected components. This matters because a finite scan can produce disconnected confidence sets; silently replacing them by the convex hull would overstate their length and change the procedure.

These pseudo-datasets are an interval-geometry illustration after the final audit, not additional audit evidence.


In [ ]:
OBSERVED_TRUTH_MUS = np.asarray([0.25, 0.75, 1.0, 1.75, 2.50])
INVERSION_MU_POINTS = 401 if FAST_MODE else 1_001
INVERSION_MU_GRID = np.linspace(*MU_RANGE, INVERSION_MU_POINTS)
observed_rng = np.random.default_rng(SEED + 1600)
observed_mean = (
    OBSERVED_TRUTH_MUS[:, None]
    * LAM_SIG
    * SIM_AUDIT_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * SIM_AUDIT_BACKGROUND_PROBABILITY[None, :]
)
observed_counts = observed_rng.poisson(observed_mean)


def response_baseline_accepts_95(mu, t_mu):
    """Endpoint-aware response-matched hNDE rule before PIT calibration."""
    mu, t_mu = np.broadcast_arrays(
        np.asarray(mu, dtype=np.float64),
        np.asarray(t_mu, dtype=np.float64),
    )
    accepted = np.empty(mu.shape, dtype=bool)
    assigned = np.zeros(mu.shape, dtype=bool)
    interior = (mu > MU_RANGE[0]) & (mu < MU_RANGE[1])
    if np.any(interior):
        u0, _ = evaluate_calibrated_pit(mu[interior], t_mu[interior])
        accepted[interior] = u0 <= .95
        assigned[interior] = True
    for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
        endpoint = mu == anchor_mu
        accepted[endpoint] = (
            t_mu[endpoint]
            <= anchor_hnde_quantiles[anchor_index, index_95]
        )
        assigned[endpoint] = True
    if not np.all(assigned):
        raise ValueError("mu lies outside the calibrated design range.")
    return accepted


def confidence_set_components(grid, accepted):
    """Return midpoint-interpolated connected components of a grid set."""
    grid = np.asarray(grid, dtype=np.float64)
    accepted = np.asarray(accepted, dtype=bool)
    if grid.ndim != 1 or accepted.shape != grid.shape:
        raise ValueError("grid and accepted must be equal-length vectors.")
    padded = np.pad(accepted.astype(np.int8), (1, 1))
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1) - 1
    components = []
    for start, stop in zip(starts, stops):
        left = (
            grid[0]
            if start == 0
            else 0.5 * (grid[start - 1] + grid[start])
        )
        right = (
            grid[-1]
            if stop == len(grid) - 1
            else 0.5 * (grid[stop] + grid[stop + 1])
        )
        components.append((float(left), float(right)))
    return components


def format_components(components):
    if not components:
        return "empty"
    return " ∪ ".join(
        f"[{left:.3f}, {right:.3f}]" for left, right in components
    )


inversion_rows = []
inversion_curves = []
for observed_index, (truth_mu, counts) in enumerate(zip(
    OBSERVED_TRUTH_MUS, observed_counts
)):
    repeated_counts = np.repeat(
        counts[None, :], INVERSION_MU_POINTS, axis=0
    )
    mu_hat_scan, t_scan, _ = response_corrected_test_statistic(
        repeated_counts, INVERSION_MU_GRID
    )
    scan_interior = (
        (INVERSION_MU_GRID > MU_RANGE[0])
        & (INVERSION_MU_GRID < MU_RANGE[1])
    )
    u0_scan = np.empty(INVERSION_MU_POINTS, dtype=np.float64)
    u_cal_scan = np.empty(INVERSION_MU_POINTS, dtype=np.float64)
    u0_scan[scan_interior], u_cal_scan[scan_interior] = (
        evaluate_calibrated_pit(
            INVERSION_MU_GRID[scan_interior],
            t_scan[scan_interior],
        )
    )
    # At the exact boundaries the construction uses empirical anchor
    # laws, so use their empirical CDFs for the displayed pivots too.
    for anchor_mu in ANCHOR_MUS:
        endpoint = INVERSION_MU_GRID == anchor_mu
        endpoint_t = float(t_scan[endpoint][0])
        u0_scan[endpoint] = np.mean(
            anchor_hnde[anchor_mu]["t_mu"] <= endpoint_t
        )
        u_cal_scan[endpoint] = np.mean(
            anchor_simulator[anchor_mu]["t_mu"] <= endpoint_t
        )
    accepted_before = response_baseline_accepts_95(
        INVERSION_MU_GRID, t_scan
    )
    accepted_after = neyman_accepts_95(
        INVERSION_MU_GRID, t_scan
    )
    components_before = confidence_set_components(
        INVERSION_MU_GRID, accepted_before
    )
    components_after = confidence_set_components(
        INVERSION_MU_GRID, accepted_after
    )
    length_before = sum(
        right - left for left, right in components_before
    )
    length_after = sum(
        right - left for left, right in components_after
    )
    inversion_rows.append({
        "dataset": observed_index,
        "truth_mu": float(truth_mu),
        "surrogate_mu_hat": float(np.median(mu_hat_scan)),
        "response_baseline_set": format_components(components_before),
        "calibrated_set": format_components(components_after),
        "response_baseline_length": float(length_before),
        "calibrated_length": float(length_after),
        "calibrated_minus_baseline": float(length_after - length_before),
    })
    inversion_curves.append({
        "u0": u0_scan,
        "u_cal": u_cal_scan,
        "before": accepted_before,
        "after": accepted_after,
    })

inversion_table = pd.DataFrame(inversion_rows)
display(inversion_table.style.format({
    "truth_mu": "{:.3f}",
    "surrogate_mu_hat": "{:.3f}",
    "response_baseline_length": "{:.3f}",
    "calibrated_length": "{:.3f}",
    "calibrated_minus_baseline": "{:+.3f}",
}).hide(axis="index"))
print(
    "Mean confidence-set length, response baseline / calibrated:",
    f"{inversion_table['response_baseline_length'].mean():.4f} / "
    f"{inversion_table['calibrated_length'].mean():.4f}",
)

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.5), sharex=True, sharey=True)
for ax, truth_mu, curves in zip(
    axes.flat, OBSERVED_TRUTH_MUS, inversion_curves
):
    ax.plot(
        INVERSION_MU_GRID, curves["u0"], ls="--", lw=1.8,
        label="response baseline",
    )
    ax.plot(
        INVERSION_MU_GRID, curves["u_cal"], lw=2.0,
        label="simulator calibrated",
    )
    ax.axhline(.95, color="black", ls=":", lw=1.1)
    ax.axvline(truth_mu, color="C2", ls="-.", lw=1.1)
    ax.fill_between(
        INVERSION_MU_GRID, 0.0, 1.0,
        where=curves["after"], color="C0", alpha=.08,
    )
    ax.set_title(rf"pseudo-data truth $\mu={truth_mu:g}$")
    ax.grid(alpha=.2)
axes.flat[len(OBSERVED_TRUTH_MUS)].axis("off")
for ax in axes[-1, :]:
    ax.set_xlabel(r"tested physical $\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel("conditional CDF / pivot")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Neyman inversion before and after residual simulator calibration")
fig.tight_layout()
export_exercise11_figure(fig, "response_pit_confidence_set_inversion")
plt.show()


## Interpretation: what each layer corrected

This exercise now separates effects that were previously entangled:

- The frozen hNDE likelihood supplies a computationally convenient ordering model in its own coordinate $\nu$.
- The pseudo-truth map $g(\mu)$ removes the leading simulator-to-hNDE response displacement. It is a reparameterization of the tested hypothesis, not a claim that the learned likelihood became true.
- The response-matched hNDE spline and first ratio estimate the inexpensive reference CDF $F_{\rm H}^{(g)}$.
- The PIT ratio estimates residual simulator differences in variance, skewness, discreteness, and tails. It learns all confidence levels at once.
- The same-template internal audit isolates conditional-calibration error under the declared construction law.
- The multinomial bootstrap measures sensitivity of the fixed construction to finite construction-template fluctuations.
- The sealed event-level audit tests transfer to simulator events that never entered the construction.

The 95% confidence set for observed counts is obtained by evaluating the response-corrected statistic and inverting

$$
\mathcal C_{0.95}(\mathcal D_{\rm obs})
=\left\{\mu\in(0,3):
G\!\left(
F_{\rm H}^{(g)}(t_\mu^{(g)}(\mathcal D_{\rm obs})\mid\mu)
\mid\mu\right)
\leq0.95\right\},
$$

with explicit empirical rules at the two exact endpoints. The equivalent cutoff form is

$$
t_\mu^{(g)}(\mathcal D_{\rm obs})
\leq
\left(F_{\rm H}^{(g)}\right)^{-1}
\!\left(G^{-1}(0.95\mid\mu)\mid\mu\right).
$$

Calibration controls size; it does not guarantee power. A poor ordering model can have correct coverage yet give longer or asymmetric confidence sets. Response matching is useful precisely because it can improve the ordering before the residual size calibration is applied.


## What the finite-template study does—and does not—establish

The notebook now reports three distinct empirical targets rather than calling all of them “coverage”:

1. **Conditional calibration on the construction template.** The internal audit asks whether the learned map reproduces one fixed empirical law.
2. **Sensitivity to construction-template fluctuations.** Multinomial bootstraps perturb that fixed law and show how much its coverage changes without retraining.
3. **Transfer to unused simulator events.** The final audit uses the deterministic complement of Exercise 5's retained evaluation reservoir.

The third is the strongest honest audit available from the existing simulation, but it cannot manufacture simulator truth. Both reservoirs are finite empirical approximations to the same underlying generator. Bootstrap spread quantifies one component of that limitation; it does not prove uniform coverage over every possible simulator distribution.

Other production targets remain possible:

- average coverage under an explicitly hierarchical template law;
- uniformly conservative coverage over a specified template nuisance set;
- conditional coverage after augmenting the calibrated parameter vector with MC-statistical or modeling nuisances.

Those are different ensembles and generally different confidence procedures. They should not be inferred from the diagnostic bootstrap used here.


## Why the PIT correction still extends to high-dimensional parameters

For a physical parameter vector $\boldsymbol\theta$, the pseudo-truth response becomes a map $\boldsymbol g(\boldsymbol\theta)$ into the surrogate-likelihood coordinates. The models are

$$
q_\phi(y\mid\boldsymbol\theta),
\qquad
r_1(y,\boldsymbol\theta),
\qquad
r_{\rm cal}(u,\boldsymbol\theta).
$$

The simulator classifier input has dimension $d+1$, but its conditional normalizer remains

$$
Z_{\rm cal}(\boldsymbol\theta)
=\int_0^1r_{\rm cal}(u,\boldsymbol\theta)\,du,
$$

a one-dimensional integral. The same is true for the $t$ primitive because the ordering statistic is scalar. In high dimensions one evaluates these integrals on demand for requested parameter batches rather than tabulating a Cartesian grid.

The response map itself becomes harder: it must be identifiable, sufficiently smooth, and locally one-to-one over the inference region if it is to serve as a useful reparameterization. Nuisance directions may be included when the desired guarantee is conditional on them, or marginalized according to a precisely stated ensemble when average coverage is the target.

PIT space solves the density-scale and normalization problem; it does not abolish the simulator curse of dimensionality. Proposal coverage, interpolation, event-level independence, and an external coverage auditor remain essential.


## Further studies

1. Compare the full PIT-density correction with direct 95% conditional quantile regression, the canonical LF2I branch. The ratio learns all confidence levels; a tail-specific regressor may be more efficient for one fixed level.
2. Increase the simulator-calibration sample and study convergence of $G(u\mid\mu)$, $c_{0.95}(\mu)$, and both internal and sealed auditors.
3. Recompute $g(\mu)$ and the full calibration inside template bootstraps to study a hierarchical average-coverage construction, keeping that target distinct from robustness of the fixed construction.
4. Add explicit simulator-model nuisance coordinates and compare conditional, marginalized, and worst-case Neyman constructions.
5. Repeat the confidence-set-length comparison over a large pseudo-observed ensemble and study power, disconnected-set frequency, and design-range truncation.
6. If this sealed audit motivates another method change, generate a new independent simulator-event reservoir before making a fresh external coverage claim.
